**Problem statement**:
Smallholder farmers may struggle to identify maize diseases early. The application would allow a user to upload or photograph a maize leaf and receive:

*   The predicted disease
*   Prediction confidence
*   A visual explanation showing the areas influencing the prediction
*   General disease information
*   A recomendation to seek confirmationfrom an agricultural specialist

A maize-focused application is more convincing than a generic 38-class plant classifier because it has a defined user, crop, problem and geographical relevance. Grey leaf spot and northern leaf blight are among the important maize stresses documented in Kenya

This project will start with four classes:

1.   Healthy maize leaf
2.   Common rust
3.   Grey leaf spot
4.   Northern leaf blight


In [ ]:
!pip install -q huggingface_hub datasets pillow pandas matplotlib


from google.colab import drive
drive.mount("/content/drive")


In [ ]:
# Importing libraries

from pathlib import Path
import json
import random
import shutil
import zipfile

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from huggingface_hub import hf_hub_download

from datasets import (
    ClassLabel,
    Dataset,
    DatasetDict,
    Features,
    Image,
    Value,
)


In [ ]:
# Reproducibility and folders

SEED = 42

random.seed(SEED)
np.random.seed(SEED)

PROJECT_DIR = Path(
    "/content/drive/MyDrive/maizeguard"
)

DATA_DIR = PROJECT_DIR / "data"

SOURCE_DIR = DATA_DIR / "source"
PROCESSED_DIR = DATA_DIR / "processed"
METADATA_DIR = DATA_DIR / "metadata"

MODEL_DIR = PROJECT_DIR / "models"
OUTPUT_DIR = PROJECT_DIR / "outputs"

for directory in [
    PROJECT_DIR,
    DATA_DIR,
    SOURCE_DIR,
    PROCESSED_DIR,
    METADATA_DIR,
    MODEL_DIR,
    OUTPUT_DIR,
]:
    directory.mkdir(
        parents=True,
        exist_ok=True
    )

print("Project directory:", PROJECT_DIR)
print("Source-image directory:", SOURCE_DIR)

In [ ]:
#Defining the repository and target classes

REPO_ID = "mohanty/PlantVillage"

TARGET_LABEL_NAMES = [
    "Corn_(maize)___Cercospora_leaf_spot Gray_leaf_spot",
    "Corn_(maize)___Common_rust_",
    "Corn_(maize)___Northern_Leaf_Blight",
    "Corn_(maize)___healthy",
]

CLASS_DISPLAY_NAMES = {
    "Corn_(maize)___Cercospora_leaf_spot Gray_leaf_spot":
        "Gray leaf spot",

    "Corn_(maize)___Common_rust_":
        "Common rust",

    "Corn_(maize)___Northern_Leaf_Blight":
        "Northern leaf blight",

    "Corn_(maize)___healthy":
        "Healthy",
}

TARGET_LABEL_TO_ID = {
    class_name: label_id
    for label_id, class_name
    in enumerate(TARGET_LABEL_NAMES)
}

ID_TO_DISPLAY_NAME = {
    label_id: CLASS_DISPLAY_NAMES[class_name]
    for class_name, label_id
    in TARGET_LABEL_TO_ID.items()
}

print("Project labels:")

for class_name, label_id in TARGET_LABEL_TO_ID.items():
    print(
        f"{label_id}: "
        f"{CLASS_DISPLAY_NAMES[class_name]}"
    )

In [ ]:
# Downloading the required repository files

DOWNLOAD_CACHE = "/content/huggingface_cache"

data_zip_path = hf_hub_download(
    repo_id=REPO_ID,
    repo_type="dataset",
    filename="data.zip",
    cache_dir=DOWNLOAD_CACHE,
)

train_split_path = hf_hub_download(
    repo_id=REPO_ID,
    repo_type="dataset",
    filename="splits/color_train.txt",
    cache_dir=DOWNLOAD_CACHE,
)

test_split_path = hf_hub_download(
    repo_id=REPO_ID,
    repo_type="dataset",
    filename="splits/color_test.txt",
    cache_dir=DOWNLOAD_CACHE,
)

leaf_map_path = hf_hub_download(
    repo_id=REPO_ID,
    repo_type="dataset",
    filename="leaf_grouping/leaf-map.json",
    cache_dir=DOWNLOAD_CACHE,
)

print("Downloaded files:")

print("Image archive:", data_zip_path)
print("Train split:", train_split_path)
print("Test split:", test_split_path)
print("Leaf map:", leaf_map_path)

In [ ]:
#Had previously run an extraction including all leaves in the collection but the
#images were so many so had to delete those that did not fit in the previously
#outlined 4 categories

import shutil

if SOURCE_DIR.exists():
    shutil.rmtree(SOURCE_DIR)

SOURCE_DIR.mkdir(
    parents=True,
    exist_ok=True
)

print("Partial extraction removed.")

In [ ]:
#To confirm the number of images in each category

from pathlib import Path
from collections import Counter

TARGET_LABEL_NAMES = [
    "Corn_(maize)___Cercospora_leaf_spot Gray_leaf_spot",
    "Corn_(maize)___Common_rust_",
    "Corn_(maize)___Northern_Leaf_Blight",
    "Corn_(maize)___healthy",
]


def read_split_file(split_file_path):
    with open(
        split_file_path,
        mode="r",
        encoding="utf-8"
    ) as file:
        return [
            line.strip()
            for line in file
            if line.strip()
        ]


train_paths = read_split_file(
    train_split_path
)

test_paths = read_split_file(
    test_split_path
)

all_split_paths = train_paths + test_paths


def identify_target_class(relative_path):
    path_parts = Path(relative_path).parts

    for class_name in TARGET_LABEL_NAMES:
        if class_name in path_parts:
            return class_name

    return None


target_path_records = []

for relative_path in all_split_paths:
    class_name = identify_target_class(
        relative_path
    )

    if class_name is not None:
        target_path_records.append({
            "relative_path": relative_path,
            "class_name": class_name,
        })


class_counts = Counter(
    record["class_name"]
    for record in target_path_records
)

print("Maize images referenced by split files:")
print("-" * 65)

for class_name in TARGET_LABEL_NAMES:
    print(
        f"{class_name}: "
        f"{class_counts[class_name]:,}"
    )

print("-" * 65)

print(
    "Total maize images:",
    f"{len(target_path_records):,}"
)

In [ ]:
# Extracting the image archive

from pathlib import Path
import zipfile

EXTRACTION_MARKER = (
    SOURCE_DIR
    / ".maize_color_extraction_complete"
)

TARGET_CLASS_SET = set(
    TARGET_LABEL_NAMES
)


def should_extract_archive_member(
    archive_member_name
):
    """
    Return True only for colour images belonging
    to one of the four target maize classes.
    """

    member_path = Path(
        archive_member_name
    )

    path_parts = member_path.parts

    # Skip folders.
    if archive_member_name.endswith("/"):
        return False

    # Only accept common image formats.
    valid_extensions = {
        ".jpg",
        ".jpeg",
        ".png",
    }

    if member_path.suffix.lower() not in valid_extensions:
        return False

    # Ensure the image belongs to the colour version.
    has_color_directory = any(
        part.lower() == "color"
        for part in path_parts
    )

    if not has_color_directory:
        return False

    # Ensure the image belongs to one of our
    # four maize classes.
    has_target_class = any(
        part in TARGET_CLASS_SET
        for part in path_parts
    )

    return has_target_class


if EXTRACTION_MARKER.exists():
    print(
        "Maize colour images have already "
        "been extracted."
    )

else:
    print(
        "Inspecting archive and selecting "
        "maize colour images..."
    )

    extracted_count = 0
    skipped_count = 0

    with zipfile.ZipFile(
        data_zip_path,
        mode="r"
    ) as archive:

        archive_members = archive.infolist()

        selected_members = [
            member
            for member in archive_members
            if should_extract_archive_member(
                member.filename
            )
        ]

        print(
            "Images selected for extraction:",
            f"{len(selected_members):,}"
        )

        if not selected_members:
            raise ValueError(
                "No maize colour images were found "
                "inside the archive. Inspect the "
                "archive path structure before continuing."
            )

        for number, member in enumerate(
            selected_members,
            start=1
        ):
            archive.extract(
                member,
                path=SOURCE_DIR
            )

            extracted_count += 1

            if (
                number % 500 == 0
                or number == len(selected_members)
            ):
                print(
                    f"Extracted {number:,} of "
                    f"{len(selected_members):,} images"
                )

    EXTRACTION_MARKER.write_text(
        (
            "PlantVillage maize colour "
            f"extraction complete: "
            f"{extracted_count} images."
        ),
        encoding="utf-8"
    )

    print(
        "Targeted extraction complete."
    )

    print(
        "Images extracted:",
        f"{extracted_count:,}"
    )

In [ ]:
# Locating the extracted raw data

raw_candidates = [
    path
    for path in SOURCE_DIR.rglob("raw")
    if (
        path.is_dir()
        and (path / "color").exists()
    )
]

if not raw_candidates:
    raise FileNotFoundError(
        "Could not locate the extracted "
        "'raw/color' image directory."
    )

RAW_DIR = raw_candidates[0]

DATA_ROOT = RAW_DIR.parent

print("Raw directory:", RAW_DIR)
print("Data root:", DATA_ROOT)

In [ ]:
#Loading the physical leaf mapping

with open(
    leaf_map_path,
    mode="r",
    encoding="utf-8"
) as file:
    leaf_map = json.load(file)

print(
    "Leaf-map entries:",
    len(leaf_map)
)

In [ ]:
 # Creating the leaf-ID function

def determine_leaf_id(
    class_name,
    filename,
    leaf_map,
    relative_path # Added relative_path as an argument
):
    """
    Determine the physical leaf identifier
    from a PlantVillage filename.
    """

    image_identifier = filename.replace(
        "_final_masked",
        ""
    )

    if "___" in image_identifier:
        image_identifier = (
            image_identifier
            .split("___")[-1]
        )

    image_identifier = (
        image_identifier
        .split("copy")[0]
    )

    for extension in [
        ".jpg",
        ".JPG",
        ".jpeg",
        ".JPEG",
        ".png",
        ".PNG",
    ]:
        image_identifier = (
            image_identifier
            .replace(extension, "")
        )

    image_identifier = image_identifier.strip()

    lookup_key = (
        image_identifier
        .lower()
        .strip()
    )

    suggestions = leaf_map.get(
        lookup_key
    )

    if not suggestions:
        # Use relative_path to ensure unique fallback IDs for distinct images
        return f"fallback_{relative_path}"

    if isinstance(suggestions, str):
        suggestions = [suggestions]

    if len(suggestions) == 1:
        return suggestions[0]

    for suggestion in suggestions:
        if class_name in suggestion:
            return suggestion

    # Use relative_path to ensure unique fallback IDs for distinct images
    return f"fallback_{relative_path}"

In [ ]:
# Function for reading a split file

def create_maize_records(
    split_file_path,
    data_root,
    leaf_map
):
    """
    Read a PlantVillage split file and
    return records for the four maize classes.
    """

    records = []
    missing_files = []

    with open(
        split_file_path,
        mode="r",
        encoding="utf-8"
    ) as file:
        relative_paths = [
            line.strip()
            for line in file
            if line.strip()
        ]

    for relative_path in relative_paths:
        path_parts = Path(
            relative_path
        ).parts

        # Expected:
        # raw/color/class_name/filename
        if len(path_parts) < 4:
            continue

        class_name = path_parts[2]
        filename = path_parts[-1]

        if class_name not in TARGET_LABEL_TO_ID:
            continue

        full_image_path = (
            data_root / relative_path
        )

        if not full_image_path.exists():
            missing_files.append(
                str(full_image_path)
            )
            continue

        class_parts = class_name.split(
            "___",
            maxsplit=1
        )

        crop = class_parts[0]

        disease = (
            class_parts[1]
            if len(class_parts) > 1
            else "unknown"
        )

        leaf_id = determine_leaf_id(
            class_name=class_name,
            filename=filename,
            leaf_map=leaf_map,
            relative_path=relative_path, # Passed relative_path
        )

        records.append({
            "image": str(full_image_path),
            "image_path": relative_path,
            "label":
                TARGET_LABEL_TO_ID[class_name],
            "crop": crop,
            "disease": disease,
            "leaf_id": leaf_id,
        })

    if missing_files:
        print(
            f"Warning: {len(missing_files)} "
            "referenced images were not found."
        )

        print(
            "First missing path:",
            missing_files[0]
        )

    return records

In [ ]:
#Building training and testing records

train_records = create_maize_records(
    split_file_path=train_split_path,
    data_root=DATA_ROOT,
    leaf_map=leaf_map,
)

test_records = create_maize_records(
    split_file_path=test_split_path,
    data_root=DATA_ROOT,
    leaf_map=leaf_map,
)

print(
    "Maize training records:",
    len(train_records)
)

print(
    "Maize testing records:",
    len(test_records)
)

if not train_records:
    raise ValueError(
        "No maize training images were found."
    )

if not test_records:
    raise ValueError(
        "No maize testing images were found."
    )

In [ ]:
# Defining the intended data set schema

dataset_features = Features({
    "image": Image(),
    "image_path": Value("string"),

    "label": ClassLabel(
        names=TARGET_LABEL_NAMES
    ),

    "crop": Value("string"),
    "disease": Value("string"),
    "leaf_id": Value("string"),
})

print(dataset_features)

In [ ]:
# Creating hugging face datasets

maize_train_pool = Dataset.from_list(
    train_records,
    features=dataset_features,
)

maize_test = Dataset.from_list(
    test_records,
    features=dataset_features,
)

print(maize_train_pool)
print(maize_test)

In [ ]:
# Verifying the schema

required_columns = {
    "image",
    "image_path",
    "label",
    "crop",
    "disease",
    "leaf_id",
}

available_columns = set(
    maize_train_pool.column_names
)

missing_columns = (
    required_columns
    - available_columns
)

if missing_columns:
    raise ValueError(
        "Required columns are still missing: "
        f"{sorted(missing_columns)}"
    )

print("Dataset structure verified.")

print(
    "Available columns:",
    maize_train_pool.column_names
)

In [ ]:
# Verifying numerial labels

print(
    "Label names:",
    maize_train_pool
    .features["label"]
    .names
)

print(
    "Labels present:",
    sorted(
        set(
            maize_train_pool["label"]
        )
    )
)

In [ ]:
# Creating a leaf grouped validation split

train_metadata = pd.DataFrame({
    "leaf_id":
        maize_train_pool["leaf_id"],

    "label":
        maize_train_pool["label"],
})

labels_per_leaf = (
    train_metadata
    .groupby("leaf_id")["label"]
    .nunique()
)

assert labels_per_leaf.max() == 1, (
    "At least one leaf_id appears "
    "under multiple disease labels."
)

unique_leaves = (
    train_metadata
    .drop_duplicates("leaf_id")
    .copy()
)

VALIDATION_FRACTION = 0.15

rng = np.random.default_rng(SEED)

validation_leaf_ids = set()

for label_id, group in (
    unique_leaves.groupby("label")
):
    leaf_ids = group[
        "leaf_id"
    ].to_numpy(copy=True)

    rng.shuffle(leaf_ids)

    validation_count = max(
        1,
        int(
            round(
                len(leaf_ids)
                * VALIDATION_FRACTION
            )
        ),
    )

    validation_leaf_ids.update(
        leaf_ids[:validation_count]
    )

print(
    "Unique training-pool leaves:",
    len(unique_leaves)
)

print(
    "Validation leaves selected:",
    len(validation_leaf_ids)
)

In [ ]:
# Creating the three splits

maize_validation = (
    maize_train_pool.filter(
        lambda leaf_id:
            leaf_id in validation_leaf_ids,

        input_columns=["leaf_id"],

        desc="Creating validation split",
    )
)

maize_train = (
    maize_train_pool.filter(
        lambda leaf_id:
            leaf_id not in validation_leaf_ids,

        input_columns=["leaf_id"],

        desc="Creating training split",
    )
)

maize_dataset = DatasetDict({
    "train": maize_train,
    "validation": maize_validation,
    "test": maize_test,
})

print(maize_dataset)

In [ ]:
# Checking for leakage

maize_dataset = DatasetDict({
    "train": maize_train,
    "validation": maize_validation,
    "test": maize_test,
})

required_splits = {
    "train",
    "validation",
    "test",
}

if "maize_dataset" not in globals():
    raise NameError(
        "maize_dataset has not been created. "
        "Run the dataset-splitting cells first."
    )

missing_splits = required_splits - set(
    maize_dataset.keys()
)

if missing_splits:
    raise ValueError(
        "Missing dataset splits: "
        f"{sorted(missing_splits)}"
    )

leaf_sets = {
    split_name: set(
        maize_dataset[split_name]["leaf_id"]
    )
    for split_name in required_splits
}

train_validation_overlap = (
    leaf_sets["train"]
    & leaf_sets["validation"]
)

train_test_overlap = (
    leaf_sets["train"]
    & leaf_sets["test"]
)

validation_test_overlap = (
    leaf_sets["validation"]
    & leaf_sets["test"]
)

print(
    "Train-validation overlapping leaves:",
    len(train_validation_overlap)
)

print(
    "Train-test overlapping leaves:",
    len(train_test_overlap)
)

print(
    "Validation-test overlapping leaves:",
    len(validation_test_overlap)
)

if train_validation_overlap:
    print(
        "Example train-validation overlaps:",
        list(train_validation_overlap)[:5]
    )

if train_test_overlap:
    print(
        "Example train-test overlaps:",
        list(train_test_overlap)[:5]
    )

if validation_test_overlap:
    print(
        "Example validation-test overlaps:",
        list(validation_test_overlap)[:5]
    )

if (
    train_validation_overlap
    or train_test_overlap
    or validation_test_overlap
):
    raise ValueError(
        "Data leakage detected: at least one "
        "leaf_id appears in multiple splits."
    )

print(
    "Leakage check passed: "
    "no leaf_id appears in multiple splits."
)

In [ ]:
#Since data leakage is detected

def get_class_leaf_groups(split_name):
    return {
        (int(label), str(leaf_id))
        for label, leaf_id in zip(
            maize_dataset[split_name]["label"],
            maize_dataset[split_name]["leaf_id"],
        )
    }


class_leaf_sets = {
    split_name: get_class_leaf_groups(split_name)
    for split_name in ["train", "validation", "test"]
}

train_validation_overlap = (
    class_leaf_sets["train"]
    & class_leaf_sets["validation"]
)

train_test_overlap = (
    class_leaf_sets["train"]
    & class_leaf_sets["test"]
)

validation_test_overlap = (
    class_leaf_sets["validation"]
    & class_leaf_sets["test"]
)

print(
    "Train-validation genuine overlap:",
    len(train_validation_overlap)
)

print(
    "Train-test genuine overlap:",
    len(train_test_overlap)
)

print(
    "Validation-test genuine overlap:",
    len(validation_test_overlap)
)

print(
    "Examples:",
    list(
        train_validation_overlap
        | train_test_overlap
        | validation_test_overlap
    )[:10]
)

In [ ]:
#Since clearly there is a lot of overlap the leaf_group_id needs to be rebuilt

train_records
test_records
from datasets import (
    ClassLabel,
    Dataset,
    DatasetDict,
    Features,
    Image,
    Value,
)

# Combining and duplicating existing records
all_records = (
    train_records
    + test_records
)

unique_records = []
seen_image_paths = set()

for record in all_records:
    image_path = record["image_path"]

    if image_path in seen_image_paths:
        continue

    seen_image_paths.add(image_path)

    corrected_record = record.copy()

    corrected_record["leaf_group_id"] = (
        f"{int(record['label'])}"
        f"::{record['leaf_id']}"
    )

    unique_records.append(
        corrected_record
    )

print(
    "Combined records:",
    len(all_records)
)

print(
    "Unique records:",
    len(unique_records)
)

# Defining the corrected features
corrected_features = Features({
    "image": Image(),
    "image_path": Value("string"),

    "label": ClassLabel(
        names=TARGET_LABEL_NAMES
    ),

    "crop": Value("string"),
    "disease": Value("string"),
    "leaf_id": Value("string"),
    "leaf_group_id": Value("string"),
})

# Creating one complete maize dataset
full_maize_dataset = Dataset.from_list(
    unique_records,
    features=corrected_features,
)

print(full_maize_dataset)

# Confirming each leaf belongs to one class
metadata = pd.DataFrame({
    "label":
        full_maize_dataset["label"],

    "leaf_id":
        full_maize_dataset["leaf_id"],

    "leaf_group_id":
        full_maize_dataset["leaf_group_id"],
})

labels_per_group = (
    metadata
    .groupby("leaf_group_id")["label"]
    .nunique()
)

assert labels_per_group.max() == 1, (
    "A leaf group is associated with "
    "more than one class."
)

print(
    "Leaf-group label consistency passed."
)

# Creating a grouped 68/12/20 split ie approx. 68% for training, 12% for
#validation and 20% for testing
SEED = 42

rng = np.random.default_rng(SEED)

unique_groups = (
    metadata[
        [
            "leaf_group_id",
            "label",
        ]
    ]
    .drop_duplicates()
    .copy()
)

train_group_ids = set()
validation_group_ids = set()
test_group_ids = set()

for label_id, class_groups in (
    unique_groups.groupby("label")
):
    group_ids = class_groups[
        "leaf_group_id"
    ].to_numpy(copy=True)

    rng.shuffle(group_ids)

    total_groups = len(group_ids)

    test_count = max(
        1,
        int(round(total_groups * 0.20))
    )

    validation_count = max(
        1,
        int(round(total_groups * 0.12))
    )

    test_end = test_count

    validation_end = (
        test_count
        + validation_count
    )

    test_group_ids.update(
        group_ids[:test_end]
    )

    validation_group_ids.update(
        group_ids[
            test_end:validation_end
        ]
    )

    train_group_ids.update(
        group_ids[validation_end:]
    )

print(
    "Training leaf groups:",
    len(train_group_ids)
)

print(
    "Validation leaf groups:",
    len(validation_group_ids)
)

print(
    "Testing leaf groups:",
    len(test_group_ids)
)

#Verifying the group assignments before filtering
assert train_group_ids.isdisjoint(
    validation_group_ids
)

assert train_group_ids.isdisjoint(
    test_group_ids
)

assert validation_group_ids.isdisjoint(
    test_group_ids
)

all_assigned_groups = (
    train_group_ids
    | validation_group_ids
    | test_group_ids
)

assert all_assigned_groups == set(
    unique_groups["leaf_group_id"]
)

print(
    "Leaf-group assignment check passed."
)

#Creating the new dataset
maize_train = full_maize_dataset.filter(
    lambda leaf_group_id:
        leaf_group_id in train_group_ids,

    input_columns=["leaf_group_id"],

    desc="Creating leakage-safe training split",
)

maize_validation = (
    full_maize_dataset.filter(
        lambda leaf_group_id:
            leaf_group_id
            in validation_group_ids,

        input_columns=["leaf_group_id"],

        desc=(
            "Creating leakage-safe "
            "validation split"
        ),
    )
)

maize_test = full_maize_dataset.filter(
    lambda leaf_group_id:
        leaf_group_id in test_group_ids,

    input_columns=["leaf_group_id"],

    desc="Creating leakage-safe test split",
)

maize_dataset = DatasetDict({
    "train": maize_train,
    "validation": maize_validation,
    "test": maize_test,
})

print(maize_dataset)

# Running the final leakage check
leaf_group_sets = {
    split_name: set(
        split_dataset["leaf_group_id"]
    )
    for split_name, split_dataset
    in maize_dataset.items()
}

train_validation_overlap = (
    leaf_group_sets["train"]
    & leaf_group_sets["validation"]
)

train_test_overlap = (
    leaf_group_sets["train"]
    & leaf_group_sets["test"]
)

validation_test_overlap = (
    leaf_group_sets["validation"]
    & leaf_group_sets["test"]
)

print(
    "Train-validation overlap:",
    len(train_validation_overlap)
)

print(
    "Train-test overlap:",
    len(train_test_overlap)
)

print(
    "Validation-test overlap:",
    len(validation_test_overlap)
)

assert not train_validation_overlap
assert not train_test_overlap
assert not validation_test_overlap

print(
    "Leakage check passed: all physical-leaf "
    "groups are isolated to one split."
)

In [ ]:
# Creating the class summary

summary_rows = []

for split_name, split_dataset in (
    maize_dataset.items()
):
    split_frame = pd.DataFrame({
        "leaf_id":
            split_dataset["leaf_id"],

        "label":
            split_dataset["label"],
    })

    for label_id, group in (
        split_frame.groupby("label")
    ):
        summary_rows.append({
            "split": split_name,

            "label":
                int(label_id),

            "class_name":
                ID_TO_DISPLAY_NAME[
                    int(label_id)
                ],

            "image_count":
                len(group),

            "unique_leaf_count":
                group["leaf_id"].nunique(),
        })

class_summary = (
    pd.DataFrame(summary_rows)
    .sort_values(
        ["split", "label"]
    )
    .reset_index(drop=True)
)

class_summary

In [ ]:
# Displaying sample images

fig, axes = plt.subplots(
    1,
    4,
    figsize=(16, 4)
)

for label_id, axis in enumerate(axes):
    matching_index = next(
        index
        for index, label
        in enumerate(
            maize_dataset["train"]["label"]
        )
        if label == label_id
    )

    sample = maize_dataset["train"][
        matching_index
    ]

    axis.imshow(sample["image"])

    axis.set_title(
        ID_TO_DISPLAY_NAME[label_id]
    )

    axis.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
# Saving the dataset

DATASET_SAVE_PATH = (
    PROCESSED_DIR
    / "plantvillage_maize_hf"
)

SUMMARY_SAVE_PATH = (
    METADATA_DIR
    / "class_summary.csv"
)

CONFIG_SAVE_PATH = (
    METADATA_DIR
    / "dataset_config.json"
)

if DATASET_SAVE_PATH.exists():
    shutil.rmtree(DATASET_SAVE_PATH)

maize_dataset.save_to_disk(
    str(DATASET_SAVE_PATH)
)

class_summary.to_csv(
    SUMMARY_SAVE_PATH,
    index=False
)

dataset_config = {
    "seed": SEED,

    "validation_fraction":
        VALIDATION_FRACTION,

    "source":
        "mohanty/PlantVillage",

    "loading_method":
        "manual repository reconstruction",

    "classes":
        TARGET_LABEL_NAMES,

    "display_names":
        ID_TO_DISPLAY_NAME,

    "split_sizes": {
        split_name:
            len(split_dataset)

        for split_name, split_dataset
        in maize_dataset.items()
    },
}

with open(
    CONFIG_SAVE_PATH,
    mode="w",
    encoding="utf-8"
) as file:
    json.dump(
        dataset_config,
        file,
        indent=2
    )

print(
    "Dataset saved to:",
    DATASET_SAVE_PATH
)

print(
    "Summary saved to:",
    SUMMARY_SAVE_PATH
)

print(
    "Configuration saved to:",
    CONFIG_SAVE_PATH
)


***Phase 2: Data Cleaning and Exploratory Analysis***

After completing the first phase, the second phase of data cleaing and
exploratory analysis begins.
The class proportions are similar across the three splits. However, Gray leaf spot represents only about 13% of the images, while Common rust and Healthy each represent about 30%. Healthy also has almost two images per physical leaf, whereas Common rust has one image per leaf. Therefore, Phase 2 must analyse both image counts and physical-leaf counts.

In [ ]:
# Identifying the physical leaf grouping column
# Identify columns available in every split

common_columns = set.intersection(
    *[
        set(split_dataset.column_names)
        for split_dataset in maize_dataset.values()
    ]
)

print("Columns found in every split:")
print(sorted(common_columns))


# Identify the physical-leaf grouping column

if "leaf_group_id" in common_columns:
    GROUP_COLUMN = "leaf_group_id"

elif "leaf_id" in common_columns:
    GROUP_COLUMN = "leaf_id"

else:
    raise KeyError(
        "Neither 'leaf_group_id' nor 'leaf_id' "
        "was found in every split."
    )


# class_name is not required as a dataset column.
# It will be reconstructed from the label.

REQUIRED_COLUMNS = {
    "image",
    "label",
    GROUP_COLUMN,
}


missing_columns = (
    REQUIRED_COLUMNS
    - common_columns
)


if missing_columns:
    raise KeyError(
        f"Required columns are missing: "
        f"{sorted(missing_columns)}"
    )


# Confirmed from the Phase 1 class summary

LABEL_TO_CLASS = {
    0: "Gray leaf spot",
    1: "Common rust",
    2: "Northern leaf blight",
    3: "Healthy",
}


print(
    f"\nPhysical-leaf grouping column: "
    f"{GROUP_COLUMN}"
)

print("\nLabel mapping:")

for label, class_name in LABEL_TO_CLASS.items():
    print(f"{label}: {class_name}")


for split_name, split_dataset in maize_dataset.items():
    print(
        f"\n{split_name}: "
        f"{split_dataset.column_names}"
    )

In [ ]:
# Rebuilding the structural manifest

manifest_parts = []


for split_name, split_dataset in maize_dataset.items():

    labels = [
        int(label)
        for label in split_dataset["label"]
    ]

    class_names = [
        LABEL_TO_CLASS[label]
        for label in labels
    ]

    split_manifest = pd.DataFrame({
        "split": split_name,

        "row_index": np.arange(
            len(split_dataset)
        ),

        "label": labels,

        "class_name": class_names,

        GROUP_COLUMN: split_dataset[
            GROUP_COLUMN
        ],
    })

    manifest_parts.append(
        split_manifest
    )


image_manifest = pd.concat(
    manifest_parts,
    ignore_index=True,
)


print(
    f"Manifest rows: "
    f"{len(image_manifest):,}"
)

display(image_manifest.head())

In [ ]:
# Validating labels, physical leaf groups and leakage
# Check that every project label maps to only
# one class name.

class_names_per_label = (
    image_manifest
    .groupby("label")["class_name"]
    .nunique()
)

if class_names_per_label.max() != 1:
    raise ValueError(
        "At least one numerical label maps to "
        "multiple class names."
    )


# Check that every physical leaf belongs to
# only one class.

labels_per_group = (
    image_manifest
    .groupby(GROUP_COLUMN)["label"]
    .nunique()
)

conflicting_groups = labels_per_group[
    labels_per_group > 1
]

if not conflicting_groups.empty:
    raise ValueError(
        f"{len(conflicting_groups)} physical-leaf "
        "groups are associated with multiple labels."
    )


# Repeat the cross-split leakage check.

group_sets = {
    split_name: set(
        split_dataset[GROUP_COLUMN]
    )
    for split_name, split_dataset
    in maize_dataset.items()
}


train_validation_overlap = (
    group_sets["train"]
    & group_sets["validation"]
)

train_test_overlap = (
    group_sets["train"]
    & group_sets["test"]
)

validation_test_overlap = (
    group_sets["validation"]
    & group_sets["test"]
)


print(
    "Train-validation overlap:",
    len(train_validation_overlap),
)

print(
    "Train-test overlap:",
    len(train_test_overlap),
)

print(
    "Validation-test overlap:",
    len(validation_test_overlap),
)


if (
    train_validation_overlap
    or train_test_overlap
    or validation_test_overlap
):
    raise ValueError(
        "Physical-leaf leakage was detected."
    )


print(
    "\nStructural checks passed: labels are "
    "consistent and physical-leaf groups remain "
    "isolated to one split."
)

In [ ]:
# Recreating the class summary
class_summary = (
    image_manifest
    .groupby(
        [
            "split",
            "label",
            "class_name",
        ],
        as_index=False,
    )
    .agg(
        image_count=(
            "row_index",
            "size",
        ),
        unique_leaf_count=(
            GROUP_COLUMN,
            "nunique",
        ),
    )
    .sort_values(
        [
            "split",
            "label",
        ]
    )
    .reset_index(drop=True)
)


class_summary["images_per_leaf"] = (
    class_summary["image_count"]
    / class_summary["unique_leaf_count"]
)


display(class_summary)

In [ ]:
#Defining image audit-functions

from PIL import Image, ImageOps
import cv2
import hashlib

def calculate_difference_hash(
    grayscale_image,
    hash_size=8,
):
    """
    Calculate a simple difference hash.

    Images with the same difference hash are
    candidates for visual review, but they are
    not automatically treated as duplicates.
    """

    resized = grayscale_image.resize(
        (
            hash_size + 1,
            hash_size,
        ),
        Image.Resampling.LANCZOS,
    )

    pixel_values = np.asarray(
        resized,
        dtype=np.uint8,
    )

    differences = (
        pixel_values[:, 1:]
        > pixel_values[:, :-1]
    )

    hash_value = 0

    for difference in differences.flatten():
        hash_value = (
            hash_value << 1
        ) | int(difference)

    hexadecimal_length = (
        hash_size
        * hash_size
        // 4
    )

    return format(
        hash_value,
        f"0{hexadecimal_length}x",
    )


def calculate_exact_pixel_hash(
    rgb_array,
):
    """
    Hash the decoded RGB pixels and their shape.

    This identifies exact visual copies even if
    their original file encodings differ.
    """

    hash_builder = hashlib.sha256()

    hash_builder.update(
        np.asarray(
            rgb_array.shape,
            dtype=np.int32,
        ).tobytes()
    )

    hash_builder.update(
        rgb_array.tobytes()
    )

    return hash_builder.hexdigest()


def audit_image(
    image,
):
    """
    Decode one image and calculate structural,
    colour and quality measurements.
    """

    image = ImageOps.exif_transpose(image)

    # Force complete decoding.
    image.load()

    original_mode = image.mode

    rgb_image = image.convert("RGB")
    grayscale_image = rgb_image.convert("L")

    rgb_array = np.asarray(
        rgb_image,
        dtype=np.uint8,
    )

    grayscale_array = np.asarray(
        grayscale_image,
        dtype=np.uint8,
    )

    hsv_array = cv2.cvtColor(
        rgb_array,
        cv2.COLOR_RGB2HSV,
    )

    laplacian = cv2.Laplacian(
        grayscale_array,
        cv2.CV_64F,
    )

    width, height = rgb_image.size

    return {
        "original_mode": original_mode,
        "width": int(width),
        "height": int(height),
        "aspect_ratio": (
            float(width / height)
            if height > 0
            else np.nan
        ),
        "brightness_mean": float(
            grayscale_array.mean()
        ),
        "contrast_std": float(
            grayscale_array.std()
        ),
        "sharpness_laplacian": float(
            laplacian.var()
        ),
        "red_mean": float(
            rgb_array[:, :, 0].mean()
        ),
        "green_mean": float(
            rgb_array[:, :, 1].mean()
        ),
        "blue_mean": float(
            rgb_array[:, :, 2].mean()
        ),
        "saturation_mean": float(
            hsv_array[:, :, 1].mean()
        ),
        "very_dark_fraction": float(
            (
                grayscale_array < 10
            ).mean()
        ),
        "very_bright_fraction": float(
            (
                grayscale_array > 245
            ).mean()
        ),
        "pixel_sha256": (
            calculate_exact_pixel_hash(
                rgb_array
            )
        ),
        "difference_hash": (
            calculate_difference_hash(
                grayscale_image
            )
        ),
    }

In [ ]:
# Auditing every image

from tqdm.auto import tqdm

audit_rows = []


for split_name, split_dataset in maize_dataset.items():

    labels = split_dataset["label"]

    group_ids = split_dataset[
        GROUP_COLUMN
    ]

    for row_index in tqdm(
        range(len(split_dataset)),
        desc=f"Auditing {split_name}",
    ):

        label = int(
            labels[row_index]
        )

        audit_record = {
            "split": split_name,
            "row_index": row_index,
            "label": label,

            # Reconstruct the class name from
            # the numerical label.
            "class_name": LABEL_TO_CLASS[
                label
            ],

            GROUP_COLUMN: (
                group_ids[row_index]
            ),
        }

        try:
            image = split_dataset[
                row_index
            ]["image"]

            image_metrics = audit_image(
                image
            )

            audit_record.update(
                image_metrics
            )

            audit_record.update({
                "decode_ok": True,
                "error_type": None,
                "error_message": None,
            })

        except Exception as error:

            audit_record.update({
                "decode_ok": False,
                "error_type": (
                    type(error).__name__
                ),
                "error_message": str(error),
            })

        audit_rows.append(
            audit_record
        )


image_audit = pd.DataFrame(
    audit_rows
)


print(
    f"Audited images: "
    f"{len(image_audit):,}"
)

display(image_audit.head())

In [ ]:
#Saving and summarizing the audit
AUDIT_SAVE_PATH = (
    METADATA_DIR
    / "phase2_image_quality_audit.csv"
)


image_audit.to_csv(
    AUDIT_SAVE_PATH,
    index=False,
)


decoding_summary = (
    image_audit
    .groupby(
        "split",
        as_index=False,
    )
    .agg(
        total_images=(
            "row_index",
            "size",
        ),
        successfully_decoded=(
            "decode_ok",
            "sum",
        ),
    )
)


decoding_summary[
    "decoding_failures"
] = (
    decoding_summary["total_images"]
    - decoding_summary[
        "successfully_decoded"
    ]
)


display(decoding_summary)

print(
    f"Audit saved to:\n"
    f"{AUDIT_SAVE_PATH}"
)


decoding_failures = image_audit[
    ~image_audit["decode_ok"]
].copy()


if decoding_failures.empty:
    print(
        "\nNo image-decoding failures detected."
    )
else:
    print(
        f"\nDecoding failures detected: "
        f"{len(decoding_failures)}"
    )

    display(decoding_failures)

In [ ]:
#Checking exact duplicates and conflicting labels
valid_image_audit = image_audit[
    image_audit["decode_ok"]
].copy()


exact_hash_summary = (
    valid_image_audit
    .groupby(
        "pixel_sha256",
        as_index=False,
    )
    .agg(
        image_count=(
            "pixel_sha256",
            "size",
        ),
        split_count=(
            "split",
            "nunique",
        ),
        label_count=(
            "label",
            "nunique",
        ),
        physical_leaf_count=(
            GROUP_COLUMN,
            "nunique",
        ),
    )
)


repeated_exact_hashes = (
    exact_hash_summary[
        exact_hash_summary[
            "image_count"
        ] > 1
    ]
    .copy()
)


cross_split_exact_duplicates = (
    repeated_exact_hashes[
        repeated_exact_hashes[
            "split_count"
        ] > 1
    ]
    .copy()
)


conflicting_label_duplicates = (
    repeated_exact_hashes[
        repeated_exact_hashes[
            "label_count"
        ] > 1
    ]
    .copy()
)


within_split_exact_duplicates = (
    repeated_exact_hashes[
        repeated_exact_hashes[
            "split_count"
        ] == 1
    ]
    .copy()
)


print(
    "Repeated exact-image hashes:",
    len(repeated_exact_hashes),
)

print(
    "Cross-split exact-image hashes:",
    len(cross_split_exact_duplicates),
)

print(
    "Conflicting-label exact-image hashes:",
    len(conflicting_label_duplicates),
)

print(
    "Within-split repeated exact-image hashes:",
    len(within_split_exact_duplicates),
)

In [ ]:
#Displaying critical duplicate cases
if not cross_split_exact_duplicates.empty:

    cross_split_duplicate_rows = (
        valid_image_audit[
            valid_image_audit[
                "pixel_sha256"
            ].isin(
                cross_split_exact_duplicates[
                    "pixel_sha256"
                ]
            )
        ]
        .sort_values(
            [
                "pixel_sha256",
                "split",
            ]
        )
    )

    print(
        "CRITICAL: exact image copies were "
        "found in multiple splits."
    )

    display(
        cross_split_duplicate_rows
    )

else:
    print(
        "Passed: no exact decoded image appears "
        "in multiple splits."
    )


if not conflicting_label_duplicates.empty:

    conflicting_duplicate_rows = (
        valid_image_audit[
            valid_image_audit[
                "pixel_sha256"
            ].isin(
                conflicting_label_duplicates[
                    "pixel_sha256"
                ]
            )
        ]
        .sort_values(
            [
                "pixel_sha256",
                "label",
            ]
        )
    )

    print(
        "\nCRITICAL: identical images have "
        "different labels."
    )

    display(
        conflicting_duplicate_rows
    )

else:
    print(
        "Passed: no exact image has "
        "conflicting labels."
    )

In [ ]:
#For the exploratory analysis



import math
from IPython.display import display


SEED = 42

random.seed(SEED)
np.random.seed(SEED)


# Confirm that the required variables exist.

required_variables = [
    "maize_dataset",
    "image_audit",
    "GROUP_COLUMN",
    "LABEL_TO_CLASS",
]

missing_variables = [
    variable_name
    for variable_name in required_variables
    if variable_name not in globals()
]

if missing_variables:
    raise NameError(
        "The following required variables are missing: "
        f"{missing_variables}. Rerun the relevant earlier cells."
    )


# Create the Phase 2 output directory

if "PHASE2_OUTPUT_DIR" not in globals():

    PHASE2_OUTPUT_DIR = Path(
        "/content/drive/MyDrive/maizeguard/"
        "outputs/phase_2_data_cleaning_and_eda"
    )


PHASE2_OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


SPLIT_ORDER = [
    "train",
    "validation",
    "test",
]

CLASS_ORDER = [
    LABEL_TO_CLASS[label]
    for label in sorted(LABEL_TO_CLASS)
]


# Reconstruct class_name if it is missing.

if "class_name" not in image_audit.columns:

    image_audit["class_name"] = (
        image_audit["label"]
        .astype(int)
        .map(LABEL_TO_CLASS)
    )


# Use only successfully decoded images for EDA.

valid_image_audit = (
    image_audit[
        image_audit["decode_ok"] == True
    ]
    .copy()
)


if valid_image_audit.empty:
    raise ValueError(
        "No successfully decoded images are available "
        "for exploratory analysis."
    )


print(
    f"Total audited images: "
    f"{len(image_audit):,}"
)

print(
    f"Successfully decoded images: "
    f"{len(valid_image_audit):,}"
)

print(
    f"Decoding failures: "
    f"{len(image_audit) - len(valid_image_audit):,}"
)

print(
    f"\nPhase 2 outputs will be saved to:\n"
    f"{PHASE2_OUTPUT_DIR}"
)

In [ ]:
# Class distribution by images and physical leaves

eda_class_summary = (
    valid_image_audit
    .groupby(
        [
            "split",
            "label",
            "class_name",
        ],
        as_index=False,
    )
    .agg(
        image_count=(
            "row_index",
            "size",
        ),
        unique_leaf_count=(
            GROUP_COLUMN,
            "nunique",
        ),
    )
)


eda_class_summary["images_per_leaf"] = (
    eda_class_summary["image_count"]
    / eda_class_summary["unique_leaf_count"]
)


eda_class_summary["image_percentage"] = (
    eda_class_summary["image_count"]
    / eda_class_summary
        .groupby("split")["image_count"]
        .transform("sum")
    * 100
)


eda_class_summary["leaf_percentage"] = (
    eda_class_summary["unique_leaf_count"]
    / eda_class_summary
        .groupby("split")["unique_leaf_count"]
        .transform("sum")
    * 100
)


eda_class_summary = (
    eda_class_summary
    .sort_values(
        [
            "split",
            "label",
        ]
    )
    .reset_index(drop=True)
)


display(
    eda_class_summary.round(2)
)

In [ ]:
# Plotting image counts by class and split
image_count_pivot = (
    eda_class_summary
    .pivot(
        index="class_name",
        columns="split",
        values="image_count",
    )
    .reindex(
        index=CLASS_ORDER,
        columns=SPLIT_ORDER,
    )
    .fillna(0)
)


ax = image_count_pivot.plot(
    kind="bar",
    figsize=(11, 6),
)

ax.set_title(
    "MaizeGuard Image Distribution by Class and Split"
)

ax.set_xlabel(
    "Maize leaf condition"
)

ax.set_ylabel(
    "Number of images"
)

plt.xticks(
    rotation=25,
    ha="right",
)

plt.legend(
    title="Split"
)

plt.tight_layout()

plt.savefig(
    PHASE2_OUTPUT_DIR
    / "class_distribution_image_counts.png",
    dpi=200,
    bbox_inches="tight",
)

plt.show()

In [ ]:
# Plotting physical leaf counts
leaf_count_pivot = (
    eda_class_summary
    .pivot(
        index="class_name",
        columns="split",
        values="unique_leaf_count",
    )
    .reindex(
        index=CLASS_ORDER,
        columns=SPLIT_ORDER,
    )
    .fillna(0)
)


ax = leaf_count_pivot.plot(
    kind="bar",
    figsize=(11, 6),
)

ax.set_title(
    "Unique Physical-Leaf Distribution by Class and Split"
)

ax.set_xlabel(
    "Maize leaf condition"
)

ax.set_ylabel(
    "Number of unique physical leaves"
)

plt.xticks(
    rotation=25,
    ha="right",
)

plt.legend(
    title="Split"
)

plt.tight_layout()

plt.savefig(
    PHASE2_OUTPUT_DIR
    / "class_distribution_unique_leaves.png",
    dpi=200,
    bbox_inches="tight",
)

plt.show()

In [ ]:
# Copmaring images per physical leaf
images_per_leaf_pivot = (
    eda_class_summary
    .pivot(
        index="class_name",
        columns="split",
        values="images_per_leaf",
    )
    .reindex(
        index=CLASS_ORDER,
        columns=SPLIT_ORDER,
    )
)


ax = images_per_leaf_pivot.plot(
    kind="bar",
    figsize=(11, 6),
)

ax.set_title(
    "Average Images per Physical Leaf"
)

ax.set_xlabel(
    "Maize leaf condition"
)

ax.set_ylabel(
    "Images per physical leaf"
)

plt.xticks(
    rotation=25,
    ha="right",
)

plt.legend(
    title="Split"
)

plt.tight_layout()

plt.savefig(
    PHASE2_OUTPUT_DIR
    / "images_per_physical_leaf.png",
    dpi=200,
    bbox_inches="tight",
)

plt.show()

In [ ]:
# Image-dimension summary
dimension_summary = (
    valid_image_audit
    .groupby(
        [
            "split",
            "class_name",
        ],
        as_index=False,
    )
    .agg(
        image_count=(
            "row_index",
            "size",
        ),
        minimum_width=(
            "width",
            "min",
        ),
        median_width=(
            "width",
            "median",
        ),
        maximum_width=(
            "width",
            "max",
        ),
        minimum_height=(
            "height",
            "min",
        ),
        median_height=(
            "height",
            "median",
        ),
        maximum_height=(
            "height",
            "max",
        ),
        minimum_aspect_ratio=(
            "aspect_ratio",
            "min",
        ),
        median_aspect_ratio=(
            "aspect_ratio",
            "median",
        ),
        maximum_aspect_ratio=(
            "aspect_ratio",
            "max",
        ),
    )
)


display(
    dimension_summary.round(3)
)

In [ ]:
# Counting image resolutions
resolution_summary = (
    valid_image_audit
    .groupby(
        [
            "width",
            "height",
        ],
        as_index=False,
    )
    .size()
    .rename(
        columns={
            "size": "image_count"
        }
    )
    .sort_values(
        "image_count",
        ascending=False,
    )
    .reset_index(drop=True)
)


resolution_summary["resolution"] = (
    resolution_summary["width"]
    .astype(int)
    .astype(str)
    + " × "
    + resolution_summary["height"]
    .astype(int)
    .astype(str)
)


print(
    "Number of unique image resolutions:",
    len(resolution_summary),
)

display(
    resolution_summary.head(20)
)

In [ ]:
# Plotting the most common resolutions
top_resolutions = (
    resolution_summary
    .head(15)
    .sort_values(
        "image_count",
        ascending=True,
    )
)


plt.figure(
    figsize=(10, 7)
)

plt.barh(
    top_resolutions["resolution"],
    top_resolutions["image_count"],
)

plt.title(
    "Fifteen Most Common Image Resolutions"
)

plt.xlabel(
    "Number of images"
)

plt.ylabel(
    "Resolution"
)

plt.tight_layout()

plt.savefig(
    PHASE2_OUTPUT_DIR
    / "most_common_image_resolutions.png",
    dpi=200,
    bbox_inches="tight",
)

plt.show()

In [ ]:
# Analyzing image modes
image_mode_summary = (
    valid_image_audit
    .groupby(
        [
            "split",
            "original_mode",
        ],
        as_index=False,
    )
    .size()
    .rename(
        columns={
            "size": "image_count"
        }
    )
    .sort_values(
        [
            "split",
            "image_count",
        ],
        ascending=[
            True,
            False,
        ],
    )
)


display(image_mode_summary)

In [ ]:
# Plotting aspect-ratio distributions
plt.figure(
    figsize=(10, 6)
)


for split_name in SPLIT_ORDER:

    split_values = (
        valid_image_audit.loc[
            valid_image_audit["split"]
            == split_name,
            "aspect_ratio",
        ]
        .dropna()
    )

    plt.hist(
        split_values,
        bins=30,
        alpha=0.45,
        label=split_name,
    )


plt.title(
    "Image Aspect-Ratio Distribution"
)

plt.xlabel(
    "Width divided by height"
)

plt.ylabel(
    "Number of images"
)

plt.legend(
    title="Split"
)

plt.tight_layout()

plt.savefig(
    PHASE2_OUTPUT_DIR
    / "aspect_ratio_distribution.png",
    dpi=200,
    bbox_inches="tight",
)

plt.show()

In [ ]:
#Quality metric summary
quality_summary = (
    valid_image_audit
    .groupby(
        [
            "split",
            "class_name",
        ],
        as_index=False,
    )
    .agg(
        brightness_mean=(
            "brightness_mean",
            "mean",
        ),
        brightness_median=(
            "brightness_mean",
            "median",
        ),
        contrast_mean=(
            "contrast_std",
            "mean",
        ),
        contrast_median=(
            "contrast_std",
            "median",
        ),
        sharpness_mean=(
            "sharpness_laplacian",
            "mean",
        ),
        sharpness_median=(
            "sharpness_laplacian",
            "median",
        ),
        saturation_mean=(
            "saturation_mean",
            "mean",
        ),
        saturation_median=(
            "saturation_mean",
            "median",
        ),
        average_dark_fraction=(
            "very_dark_fraction",
            "mean",
        ),
        average_bright_fraction=(
            "very_bright_fraction",
            "mean",
        ),
    )
)


display(
    quality_summary.round(3)
)

In [ ]:
# Reusable boxplot function
training_audit = (
    valid_image_audit[
        valid_image_audit["split"]
        == "train"
    ]
    .copy()
)


def plot_training_metric_by_class(
    metric,
    title,
    y_label,
    save_name,
):
    """
    Plot one training-set metric for the four
    maize classes.
    """

    metric_values = [
        training_audit.loc[
            training_audit["class_name"]
            == class_name,
            metric,
        ]
        .dropna()
        .values

        for class_name in CLASS_ORDER
    ]

    plt.figure(
        figsize=(11, 6)
    )

    plt.boxplot(
        metric_values,
        tick_labels=CLASS_ORDER,
        showfliers=False,
    )

    plt.title(title)

    plt.xlabel(
        "Maize leaf condition"
    )

    plt.ylabel(y_label)

    plt.xticks(
        rotation=25,
        ha="right",
    )

    plt.tight_layout()

    plt.savefig(
        PHASE2_OUTPUT_DIR / save_name,
        dpi=200,
        bbox_inches="tight",
    )

    plt.show()

In [ ]:
# Brightness by calss
plot_training_metric_by_class(
    metric="brightness_mean",
    title=(
        "Training-Set Brightness Distribution "
        "by Class"
    ),
    y_label="Mean grayscale brightness",
    save_name="training_brightness_by_class.png",
)

#Contrast by class
plot_training_metric_by_class(
    metric="contrast_std",
    title=(
        "Training-Set Contrast Distribution "
        "by Class"
    ),
    y_label="Grayscale intensity standard deviation",
    save_name="training_contrast_by_class.png",
)

# Sharpness by class
plot_training_metric_by_class(
    metric="sharpness_laplacian",
    title=(
        "Training-Set Sharpness Distribution "
        "by Class"
    ),
    y_label="Laplacian variance",
    save_name="training_sharpness_by_class.png",
)

# Saturation by class
plot_training_metric_by_class(
    metric="saturation_mean",
    title=(
        "Training-Set Saturation Distribution "
        "by Class"
    ),
    y_label="Mean HSV saturation",
    save_name="training_saturation_by_class.png",
)

In [ ]:
# Display random training images from every class
def show_random_class_samples(
    split_name="train",
    samples_per_class=4,
    seed=42,
):
    """
    Display random successfully decoded images
    from each class in one dataset split.
    """

    if split_name not in maize_dataset:
        raise KeyError(
            f"Unknown split: {split_name}"
        )

    random_generator = np.random.default_rng(
        seed
    )

    figure, axes = plt.subplots(
        nrows=len(CLASS_ORDER),
        ncols=samples_per_class,
        figsize=(
            4 * samples_per_class,
            3.5 * len(CLASS_ORDER),
        ),
    )

    axes = np.asarray(axes).reshape(
        len(CLASS_ORDER),
        samples_per_class,
    )

    for class_position, class_name in enumerate(
        CLASS_ORDER
    ):

        candidates = (
            valid_image_audit[
                (
                    valid_image_audit["split"]
                    == split_name
                )
                & (
                    valid_image_audit["class_name"]
                    == class_name
                )
            ]
        )

        sample_size = min(
            samples_per_class,
            len(candidates),
        )

        selected_positions = (
            random_generator.choice(
                candidates.index.to_numpy(),
                size=sample_size,
                replace=False,
            )
        )

        selected_rows = candidates.loc[
            selected_positions
        ]

        for image_position in range(
            samples_per_class
        ):

            axis = axes[
                class_position,
                image_position,
            ]

            axis.axis("off")

            if image_position >= sample_size:
                continue

            selected_row = selected_rows.iloc[
                image_position
            ]

            row_index = int(
                selected_row["row_index"]
            )

            image = maize_dataset[
                split_name
            ][row_index]["image"]

            axis.imshow(image)

            axis.set_title(
                f"{class_name}\n"
                f"Leaf: "
                f"{selected_row[GROUP_COLUMN]}"
            )

    figure.suptitle(
        f"Random {split_name.capitalize()} Images",
        fontsize=16,
    )

    plt.tight_layout()

    plt.show()


show_random_class_samples(
    split_name="train",
    samples_per_class=4,
    seed=SEED,
)

In [ ]:
# Displaying validation samples

show_random_class_samples(
    split_name="validation",
    samples_per_class=3,
    seed=SEED,
)

In [ ]:
#Calculating review threasholds using trianing data only
review_threshold_rows = []


threshold_configuration = {
    "brightness_mean": {
        "use_lower": True,
        "use_upper": True,
    },
    "contrast_std": {
        "use_lower": True,
        "use_upper": False,
    },
    "sharpness_laplacian": {
        "use_lower": True,
        "use_upper": False,
    },
    "saturation_mean": {
        "use_lower": True,
        "use_upper": True,
    },
    "aspect_ratio": {
        "use_lower": True,
        "use_upper": True,
    },
    "very_dark_fraction": {
        "use_lower": False,
        "use_upper": True,
    },
    "very_bright_fraction": {
        "use_lower": False,
        "use_upper": True,
    },
}


for class_name in CLASS_ORDER:

    class_training_data = (
        training_audit[
            training_audit["class_name"]
            == class_name
        ]
    )

    for metric, configuration in (
        threshold_configuration.items()
    ):

        values = (
            class_training_data[metric]
            .dropna()
        )

        if values.empty:
            continue

        lower_threshold = (
            float(values.quantile(0.01))
            if configuration["use_lower"]
            else np.nan
        )

        upper_threshold = (
            float(values.quantile(0.99))
            if configuration["use_upper"]
            else np.nan
        )

        review_threshold_rows.append({
            "class_name": class_name,
            "metric": metric,
            "lower_threshold": lower_threshold,
            "upper_threshold": upper_threshold,
        })


review_thresholds = pd.DataFrame(
    review_threshold_rows
)


display(
    review_thresholds.round(5)
)

In [ ]:
# Applying the review threasholds

audit_with_flags = (
    valid_image_audit.copy()
)


flag_columns = []


for threshold_row in (
    review_thresholds.itertuples(
        index=False
    )
):

    class_name = threshold_row.class_name
    metric = threshold_row.metric

    class_mask = (
        audit_with_flags["class_name"]
        == class_name
    )


    if not pd.isna(
        threshold_row.lower_threshold
    ):

        flag_name = (
            f"flag_{metric}_low"
        )

        if flag_name not in (
            audit_with_flags.columns
        ):
            audit_with_flags[
                flag_name
            ] = False

        audit_with_flags.loc[
            class_mask
            & (
                audit_with_flags[metric]
                < threshold_row.lower_threshold
            ),
            flag_name,
        ] = True

        flag_columns.append(flag_name)


    if not pd.isna(
        threshold_row.upper_threshold
    ):

        flag_name = (
            f"flag_{metric}_high"
        )

        if flag_name not in (
            audit_with_flags.columns
        ):
            audit_with_flags[
                flag_name
            ] = False

        audit_with_flags.loc[
            class_mask
            & (
                audit_with_flags[metric]
                > threshold_row.upper_threshold
            ),
            flag_name,
        ] = True

        flag_columns.append(flag_name)


flag_columns = sorted(
    set(flag_columns)
)


audit_with_flags["review_flag_count"] = (
    audit_with_flags[
        flag_columns
    ]
    .sum(axis=1)
)


def combine_review_reasons(row):

    reasons = [
        column_name.replace(
            "flag_",
            "",
        )

        for column_name in flag_columns

        if bool(row[column_name])
    ]

    return "; ".join(reasons)


audit_with_flags["review_reasons"] = (
    audit_with_flags.apply(
        combine_review_reasons,
        axis=1,
    )
)


review_candidates = (
    audit_with_flags[
        audit_with_flags[
            "review_flag_count"
        ] > 0
    ]
    .copy()
    .sort_values(
        [
            "review_flag_count",
            "split",
            "class_name",
        ],
        ascending=[
            False,
            True,
            True,
        ],
    )
)


print(
    f"Images flagged for visual review: "
    f"{len(review_candidates):,}"
)

print(
    "These images have not been deleted."
)

In [ ]:
#Summarizing review candidates

review_summary = (
    review_candidates
    .groupby(
        [
            "split",
            "class_name",
        ],
        as_index=False,
    )
    .agg(
        flagged_images=(
            "row_index",
            "size",
        ),
        maximum_flags_on_one_image=(
            "review_flag_count",
            "max",
        ),
    )
)


total_by_split_class = (
    valid_image_audit
    .groupby(
        [
            "split",
            "class_name",
        ],
        as_index=False,
    )
    .size()
    .rename(
        columns={
            "size": "total_images"
        }
    )
)


review_summary = (
    total_by_split_class
    .merge(
        review_summary,
        on=[
            "split",
            "class_name",
        ],
        how="left",
    )
)


review_summary[
    [
        "flagged_images",
        "maximum_flags_on_one_image",
    ]
] = (
    review_summary[
        [
            "flagged_images",
            "maximum_flags_on_one_image",
        ]
    ]
    .fillna(0)
    .astype(int)
)


review_summary["flagged_percentage"] = (
    review_summary["flagged_images"]
    / review_summary["total_images"]
    * 100
)


display(
    review_summary.round(2)
)

In [ ]:
#Summarizing the most common review reasons
flag_reason_summary = pd.DataFrame({
    "review_reason": [
        column_name.replace(
            "flag_",
            "",
        )
        for column_name in flag_columns
    ],

    "image_count": [
        int(
            audit_with_flags[
                column_name
            ].sum()
        )
        for column_name in flag_columns
    ],
})


flag_reason_summary = (
    flag_reason_summary
    .sort_values(
        "image_count",
        ascending=False,
    )
    .reset_index(drop=True)
)


display(flag_reason_summary)

In [ ]:
# Displaying the most unusual training images

def show_flagged_images(
    split_name="train",
    maximum_images=20,
    seed=42,
):
    """
    Display images flagged by several review
    criteria. No images are removed.
    """

    candidates = (
        review_candidates[
            review_candidates["split"]
            == split_name
        ]
        .copy()
    )

    if candidates.empty:
        print(
            f"No review candidates found in "
            f"the {split_name} split."
        )
        return

    # Prioritise images with the largest number
    # of simultaneous flags.

    candidates = candidates.sort_values(
        "review_flag_count",
        ascending=False,
    )

    selected = candidates.head(
        maximum_images
    )

    number_of_images = len(selected)

    number_of_columns = 4

    number_of_rows = math.ceil(
        number_of_images
        / number_of_columns
    )

    figure, axes = plt.subplots(
        number_of_rows,
        number_of_columns,
        figsize=(
            4 * number_of_columns,
            4 * number_of_rows,
        ),
    )

    axes = np.asarray(
        axes
    ).reshape(-1)

    for axis in axes:
        axis.axis("off")

    for axis, (_, row) in zip(
        axes,
        selected.iterrows(),
    ):

        row_index = int(
            row["row_index"]
        )

        image = maize_dataset[
            split_name
        ][row_index]["image"]

        axis.imshow(image)

        shortened_reasons = (
            row["review_reasons"]
            .replace(
                "sharpness_laplacian",
                "sharpness",
            )
            .replace(
                "brightness_mean",
                "brightness",
            )
            .replace(
                "contrast_std",
                "contrast",
            )
            .replace(
                "saturation_mean",
                "saturation",
            )
        )

        axis.set_title(
            f"{row['class_name']}\n"
            f"Flags: {int(row['review_flag_count'])}\n"
            f"{shortened_reasons}",
            fontsize=9,
        )

        axis.axis("off")

    figure.suptitle(
        f"Flagged {split_name.capitalize()} "
        f"Images for Manual Review",
        fontsize=16,
    )

    plt.tight_layout()

    plt.show()


show_flagged_images(
    split_name="train",
    maximum_images=20,
    seed=SEED,
)

In [ ]:
# Saving all EDA tables

eda_class_summary.to_csv(
    PHASE2_OUTPUT_DIR
    / "eda_class_summary.csv",
    index=False,
)


dimension_summary.to_csv(
    PHASE2_OUTPUT_DIR
    / "eda_dimension_summary.csv",
    index=False,
)


resolution_summary.to_csv(
    PHASE2_OUTPUT_DIR
    / "eda_resolution_summary.csv",
    index=False,
)


image_mode_summary.to_csv(
    PHASE2_OUTPUT_DIR
    / "eda_image_mode_summary.csv",
    index=False,
)


quality_summary.to_csv(
    PHASE2_OUTPUT_DIR
    / "eda_quality_summary.csv",
    index=False,
)


review_thresholds.to_csv(
    PHASE2_OUTPUT_DIR
    / "training_based_review_thresholds.csv",
    index=False,
)


review_candidates.to_csv(
    PHASE2_OUTPUT_DIR
    / "images_flagged_for_manual_review.csv",
    index=False,
)


review_summary.to_csv(
    PHASE2_OUTPUT_DIR
    / "review_candidate_summary.csv",
    index=False,
)


flag_reason_summary.to_csv(
    PHASE2_OUTPUT_DIR
    / "review_reason_summary.csv",
    index=False,
)


print(
    "Phase 2B tables and visualisations "
    "were saved successfully."
)

print(
    f"\nOutput directory:\n"
    f"{PHASE2_OUTPUT_DIR}"
)

In [ ]:
#Phase 2 checkpoint
print("=" * 65)
print("PHASE 2B: EXPLORATORY DATA ANALYSIS CHECKPOINT")
print("=" * 65)

print(
    f"\nTotal images audited: "
    f"{len(image_audit):,}"
)

print(
    f"Successfully decoded: "
    f"{len(valid_image_audit):,}"
)

print(
    f"Decoding failures: "
    f"{len(image_audit) - len(valid_image_audit):,}"
)

print(
    f"Unique image resolutions: "
    f"{len(resolution_summary):,}"
)

print(
    f"Images flagged for manual review: "
    f"{len(review_candidates):,}"
)

print(
    "\nNo images have been removed."
)

print(
    "Phase 2B is complete."
)

print(
    "The next step is Phase 2C: review the flagged "
    "images and document the final cleaning decision."
)

In [ ]:
#Set up and prerequisite check
print("=" * 65)
print("PHASE 2B: EXPLORATORY DATA ANALYSIS CHECKPOINT")
print("=" * 65)

print(
    f"\nTotal images audited: "
    f"{len(image_audit):,}"
)

print(
    f"Successfully decoded: "
    f"{len(valid_image_audit):,}"
)

print(
    f"Decoding failures: "
    f"{len(image_audit) - len(valid_image_audit):,}"
)

print(
    f"Unique image resolutions: "
    f"{len(resolution_summary):,}"
)

print(
    f"Images flagged for manual review: "
    f"{len(review_candidates):,}"
)

print(
    "\nNo images have been removed."
)

print(
    "Phase 2B is complete."
)

print(
    "The next step is Phase 2C: review the flagged "
    "images and document the final cleaning decision."
)

In [ ]:
#Creating stable review identifiers
image_audit = image_audit.copy()

image_audit["review_id"] = (
    image_audit["split"].astype(str)
    + ":"
    + image_audit["row_index"]
        .astype(int)
        .astype(str)
)


review_candidates = review_candidates.copy()

if "review_id" not in review_candidates.columns:

    review_candidates["review_id"] = (
        review_candidates["split"].astype(str)
        + ":"
        + review_candidates["row_index"]
            .astype(int)
            .astype(str)
    )


if "class_name" not in image_audit.columns:

    image_audit["class_name"] = (
        image_audit["label"]
        .astype(int)
        .map(LABEL_TO_CLASS)
    )


print(
    f"Total image records: "
    f"{len(image_audit):,}"
)

print(
    f"Flagged review candidates: "
    f"{len(review_candidates):,}"
)

display(
    review_candidates[
        [
            "review_id",
            "split",
            "class_name",
            GROUP_COLUMN,
            "review_flag_count",
            "review_reasons",
        ]
    ].head(10)
)

In [ ]:
#Repeating critical duplicate check

valid_hash_audit = (
    image_audit[
        image_audit["decode_ok"] == True
    ]
    .copy()
)


if "pixel_sha256" not in valid_hash_audit.columns:
    raise KeyError(
        "The pixel_sha256 column is missing. "
        "Rerun the image audit before continuing."
    )


hash_summary = (
    valid_hash_audit
    .groupby(
        "pixel_sha256",
        as_index=False,
    )
    .agg(
        image_count=(
            "pixel_sha256",
            "size",
        ),
        split_count=(
            "split",
            "nunique",
        ),
        label_count=(
            "label",
            "nunique",
        ),
    )
)


cross_split_duplicates = (
    hash_summary[
        hash_summary["split_count"] > 1
    ]
)

conflicting_label_duplicates = (
    hash_summary[
        hash_summary["label_count"] > 1
    ]
)


if not cross_split_duplicates.empty:
    raise ValueError(
        "Exact image copies exist in multiple splits. "
        "Resolve them before creating the cleaned dataset."
    )


if not conflicting_label_duplicates.empty:
    raise ValueError(
        "Identical images have conflicting labels. "
        "Resolve them before creating the cleaned dataset."
    )


print(
    "Passed: no exact decoded image appears "
    "in multiple splits."
)

print(
    "Passed: no exact image has "
    "conflicting labels."
)

In [ ]:
#Creating the manual review queue
manual_review_queue = (
    review_candidates
    .sort_values(
        [
            "review_flag_count",
            "split",
            "class_name",
        ],
        ascending=[
            False,
            True,
            True,
        ],
    )
    .reset_index(drop=True)
)


review_queue_columns = [
    "review_id",
    "split",
    "row_index",
    "label",
    "class_name",
    GROUP_COLUMN,
    "review_flag_count",
    "review_reasons",
    "brightness_mean",
    "contrast_std",
    "sharpness_laplacian",
    "saturation_mean",
    "aspect_ratio",
]


review_queue_columns = [
    column
    for column in review_queue_columns
    if column in manual_review_queue.columns
]


display(
    manual_review_queue[
        review_queue_columns
    ].head(20)
)


manual_review_queue.to_csv(
    PHASE2_OUTPUT_DIR
    / "manual_review_queue.csv",
    index=False,
)


print(
    "Manual-review queue saved successfully."
)

In [ ]:
#Record no removal decision

MANUAL_REMOVE_DECISIONS = {}

print(
    "Manual removals selected:",
    len(MANUAL_REMOVE_DECISIONS),
)

print(
    "Cleaning decision: retain all images."
)

In [ ]:
# Verifying that retaining all images is valid

decoding_failures = image_audit[
    image_audit["decode_ok"] == False
].copy()


if not decoding_failures.empty:

    print(
        f"WARNING: {len(decoding_failures)} images "
        "failed to decode."
    )

    display(
        decoding_failures[
            [
                "split",
                "row_index",
                "class_name",
                "error_type",
                "error_message",
            ]
        ]
    )

    raise ValueError(
        "Images that failed to decode cannot be "
        "retained. Review these images first."
    )


print(
    "Passed: all images decoded successfully."
)

print(
    "No automatic removals are required."
)

print(
    "No manual removals are required."
)

In [ ]:
# Creating and saving the final cleaning decision table

final_cleaning_decisions = (
    image_audit.copy()
)


if "review_id" not in final_cleaning_decisions.columns:

    final_cleaning_decisions["review_id"] = (
        final_cleaning_decisions[
            "split"
        ].astype(str)
        + ":"
        + final_cleaning_decisions[
            "row_index"
        ].astype(int).astype(str)
    )


final_cleaning_decisions[
    "cleaning_decision"
] = "keep"


final_cleaning_decisions[
    "decision_reason"
] = "passed_integrity_and_quality_review"


final_cleaning_decisions[
    "decision_source"
] = "phase2_review"


if "PHASE2_OUTPUT_DIR" not in globals():

    from pathlib import Path

    PHASE2_OUTPUT_DIR = Path(
        "/content/drive/MyDrive/maizeguard/"
        "outputs/phase_2_data_cleaning_and_eda"
    )


PHASE2_OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


FINAL_DECISIONS_PATH = (
    PHASE2_OUTPUT_DIR
    / "final_cleaning_decisions.csv"
)


final_cleaning_decisions.to_csv(
    FINAL_DECISIONS_PATH,
    index=False,
)


print(
    "Final cleaning decisions saved."
)

print(
    f"\nSaved to:\n{FINAL_DECISIONS_PATH}"
)

print(
    "\nDecision counts:"
)

print(
    final_cleaning_decisions[
        "cleaning_decision"
    ].value_counts()
)

In [ ]:
# Final phase 2 checkpoint

# Recheck physical-leaf isolation.

final_group_sets = {
    split_name: set(
        split_dataset[GROUP_COLUMN]
    )
    for split_name, split_dataset
    in maize_dataset.items()
}


train_validation_overlap = (
    final_group_sets["train"]
    & final_group_sets["validation"]
)

train_test_overlap = (
    final_group_sets["train"]
    & final_group_sets["test"]
)

validation_test_overlap = (
    final_group_sets["validation"]
    & final_group_sets["test"]
)


total_images = sum(
    len(split_dataset)
    for split_dataset
    in maize_dataset.values()
)


print("=" * 68)
print("PHASE 2: DATA CLEANING AND EDA — FINAL CHECKPOINT")
print("=" * 68)

print(
    f"\nTotal images reviewed: "
    f"{total_images:,}"
)

print(
    "Images removed: 0"
)

print(
    f"Images retained: "
    f"{total_images:,}"
)

print(
    "\nPhysical-leaf overlap:"
)

print(
    "  Train-validation:",
    len(train_validation_overlap),
)

print(
    "  Train-test:",
    len(train_test_overlap),
)

print(
    "  Validation-test:",
    len(validation_test_overlap),
)


if (
    train_validation_overlap
    or train_test_overlap
    or validation_test_overlap
):

    raise ValueError(
        "Physical-leaf leakage detected."
    )


print(
    "\nNo decoding failures were detected."
)

print(
    "No cross-split exact duplicates were detected."
)

print(
    "No conflicting duplicate labels were detected."
)

print(
    "Flagged statistical outliers were reviewed "
    "and retained because they were valid images."
)

print(
    "\nPHASE 2 IS COMPLETE."
)

print(
    "Use maize_dataset as the input dataset "
    "for Phase 3."
)

***Phase 3: Image Processing, Data Augmentation, and TensorFlow Pipelines***

In [ ]:

# PHASE 3
# Image preprocessing, augmentation and TensorFlow pipelines


from io import BytesIO
from collections.abc import Mapping
import re
import tensorflow as tf
import datasets as hf_datasets

from PIL import Image as PILImage
from PIL import ImageOps


# Reproducibility


SEED = globals().get("SEED", 42)

random.seed(SEED)
np.random.seed(SEED)
tf.keras.utils.set_random_seed(SEED)

try:
    tf.config.experimental.enable_op_determinism()
    DETERMINISTIC_OPERATIONS = True
except Exception:
    DETERMINISTIC_OPERATIONS = False


# Phase 3 configuration

IMAGE_HEIGHT = 224
IMAGE_WIDTH = 224
IMAGE_SIZE = (IMAGE_HEIGHT, IMAGE_WIDTH)

BATCH_SIZE = 32

AUTOTUNE = tf.data.AUTOTUNE

# Resized images are cached as uint8 rather than float32
# to reduce memory usage.
CACHE_IN_MEMORY = True

# Used only when CACHE_IN_MEMORY is False.
TF_CACHE_DIRECTORY = Path(
    "/content/maizeguard_tf_cache"
)


print("TensorFlow version:", tf.__version__)
print("Hugging Face datasets version:", hf_datasets.__version__)
print("Image size:", IMAGE_SIZE)
print("Batch size:", BATCH_SIZE)
print("Random seed:", SEED)
print(
    "Deterministic TensorFlow operations:",
    DETERMINISTIC_OPERATIONS,
)

In [ ]:
# Validating phase 1 and 2 objects


required_objects = [
    "maize_dataset",
    "GROUP_COLUMN",
    "LABEL_TO_CLASS",
]

missing_objects = [
    object_name
    for object_name in required_objects
    if object_name not in globals()
]

if missing_objects:
    raise NameError(
        "The following required notebook objects are missing: "
        f"{missing_objects}"
    )



# Confirm the required dataset splits

available_splits = list(maize_dataset.keys())

if "train" not in available_splits:
    raise KeyError(
        "maize_dataset does not contain a 'train' split."
    )

if "test" not in available_splits:
    raise KeyError(
        "maize_dataset does not contain a 'test' split."
    )

if "validation" in available_splits:
    VALIDATION_SPLIT_NAME = "validation"

elif "val" in available_splits:
    VALIDATION_SPLIT_NAME = "val"

else:
    raise KeyError(
        "maize_dataset must contain either a 'validation' "
        "or 'val' split."
    )


# Read the existing split sizes directly from maize_dataset

CURRENT_SPLIT_SIZES = {
    "train": len(maize_dataset["train"]),
    VALIDATION_SPLIT_NAME: len(
        maize_dataset[VALIDATION_SPLIT_NAME]
    ),
    "test": len(maize_dataset["test"]),
}

TOTAL_IMAGES = sum(
    CURRENT_SPLIT_SIZES.values()
)


# The completed Phase 2 audit retained 3,852 images.
if TOTAL_IMAGES != 3852:
    raise ValueError(
        "The current DatasetDict does not contain the expected "
        "3,852 retained images.\n"
        f"Current split sizes: {CURRENT_SPLIT_SIZES}\n"
        f"Current total: {TOTAL_IMAGES:,}"
    )


# Confirm that required columns exist in every split


for split_name in [
    "train",
    VALIDATION_SPLIT_NAME,
    "test",
]:
    split_columns = (
        maize_dataset[split_name].column_names
    )

    if GROUP_COLUMN not in split_columns:
        raise KeyError(
            f"GROUP_COLUMN='{GROUP_COLUMN}' is missing from "
            f"the '{split_name}' split."
        )



# Save expected sizes for later Phase 3 checks

EXPECTED_SPLIT_SIZES = (
    CURRENT_SPLIT_SIZES.copy()
)


print("Existing DatasetDict validated successfully.")
print("Available splits:", available_splits)
print(
    "Validation split name:",
    VALIDATION_SPLIT_NAME,
)
print(
    "Physical-leaf column:",
    GROUP_COLUMN,
)

print("\nCurrent split sizes:")

for split_name, split_size in (
    CURRENT_SPLIT_SIZES.items()
):
    print(
        f"  {split_name}: {split_size:,}"
    )

print(
    f"\nTotal retained images: "
    f"{TOTAL_IMAGES:,}"
)

print(
    "No dataset recreation, filtering or "
    "resplitting was performed."
)

In [ ]:
# Detecting the image and label columns and standardize the the class mapping


train_split = maize_dataset["train"]
train_features = train_split.features
train_columns = train_split.column_names

sample_row = train_split[0]


def looks_like_image_value(value):
    """
    Return True when a value appears to represent an image.
    """

    if isinstance(value, PILImage.Image):
        return True

    if isinstance(value, np.ndarray):
        return value.ndim in (2, 3)

    if isinstance(value, Mapping):
        return (
            value.get("path") is not None
            or value.get("bytes") is not None
        )

    return False


# Respect an existing IMAGE_COLUMN variable when available.
existing_image_column = globals().get(
    "IMAGE_COLUMN"
)

if (
    existing_image_column is not None
    and existing_image_column in train_columns
):
    IMAGE_COLUMN = existing_image_column
else:
    image_feature_columns = [
        column_name
        for column_name, feature in train_features.items()
        if feature.__class__.__name__ == "Image"
    ]

    if len(image_feature_columns) == 1:
        IMAGE_COLUMN = image_feature_columns[0]
    else:
        image_name_candidates = [
            "image",
            "images",
            "img",
            "pixel_values",
            "image_path",
            "file_path",
            "filepath",
        ]

        IMAGE_COLUMN = next(
            (
                column_name
                for column_name in image_name_candidates
                if column_name in train_columns
            ),
            None,
        )

        if IMAGE_COLUMN is None:
            detected_image_columns = [
                column_name
                for column_name, value in sample_row.items()
                if looks_like_image_value(value)
            ]

            if len(detected_image_columns) != 1:
                raise ValueError(
                    "Unable to identify one unambiguous image "
                    "column. Found possible columns: "
                    f"{detected_image_columns}"
                )

            IMAGE_COLUMN = detected_image_columns[0]


# Respect an existing LABEL_COLUMN variable when available.
existing_label_column = globals().get(
    "LABEL_COLUMN"
)

if (
    existing_label_column is not None
    and existing_label_column in train_columns
):
    LABEL_COLUMN = existing_label_column
else:
    class_label_columns = [
        column_name
        for column_name, feature in train_features.items()
        if feature.__class__.__name__ == "ClassLabel"
    ]

    if len(class_label_columns) == 1:
        LABEL_COLUMN = class_label_columns[0]
    else:
        label_name_candidates = [
            "label",
            "labels",
            "class_id",
            "class",
            "target",
        ]

        LABEL_COLUMN = next(
            (
                column_name
                for column_name in label_name_candidates
                if column_name in train_columns
            ),
            None,
        )

        if LABEL_COLUMN is None:
            raise ValueError(
                "Unable to identify the label column. "
                f"Available columns: {train_columns}"
            )


if IMAGE_COLUMN == LABEL_COLUMN:
    raise ValueError(
        "The detected image and label columns are identical."
    )


# Normalize LABEL_TO_CLASS to an integer-ID-to-name mapping


def is_integer_like(value):
    if isinstance(value, (int, np.integer)):
        return True

    if isinstance(value, str):
        return value.strip().lstrip("-").isdigit()

    return False


if isinstance(LABEL_TO_CLASS, (list, tuple)):
    ID_TO_CLASS = {
        index: str(class_name)
        for index, class_name in enumerate(
            LABEL_TO_CLASS
        )
    }

elif isinstance(LABEL_TO_CLASS, Mapping):
    mapping_keys = list(LABEL_TO_CLASS.keys())
    mapping_values = list(LABEL_TO_CLASS.values())

    if all(
        is_integer_like(key)
        for key in mapping_keys
    ):
        ID_TO_CLASS = {
            int(label_id): str(class_name)
            for label_id, class_name
            in LABEL_TO_CLASS.items()
        }

    elif all(
        is_integer_like(value)
        for value in mapping_values
    ):
        ID_TO_CLASS = {
            int(label_id): str(class_name)
            for class_name, label_id
            in LABEL_TO_CLASS.items()
        }

    else:
        raise ValueError(
            "LABEL_TO_CLASS must map integer IDs to class "
            "names or class names to integer IDs."
        )

else:
    raise TypeError(
        "LABEL_TO_CLASS must be a dictionary, list or tuple."
    )


ID_TO_CLASS = dict(
    sorted(ID_TO_CLASS.items())
)

expected_label_ids = set(
    range(len(ID_TO_CLASS))
)

if set(ID_TO_CLASS.keys()) != expected_label_ids:
    raise ValueError(
        "The class IDs must be consecutive integers beginning "
        f"at zero. Found: {sorted(ID_TO_CLASS.keys())}"
    )


NUM_CLASSES = len(ID_TO_CLASS)
CLASS_NAMES = [
    ID_TO_CLASS[label_id]
    for label_id in range(NUM_CLASSES)
]

CLASS_TO_LABEL = {
    class_name: label_id
    for label_id, class_name in ID_TO_CLASS.items()
}


if NUM_CLASSES != 4:
    raise ValueError(
        "MaizeGuard should contain exactly four classes, "
        f"but {NUM_CLASSES} classes were found."
    )


print("Image column:", IMAGE_COLUMN)
print("Label column:", LABEL_COLUMN)
print("Physical-leaf column:", GROUP_COLUMN)
print("Number of classes:", NUM_CLASSES)

print("\nClass mapping:")
for label_id, class_name in ID_TO_CLASS.items():
    print(f"  {label_id}: {class_name}")

In [ ]:
# Detecting the image and label columns and standardize the class mapping


train_split = maize_dataset["train"]
train_features = train_split.features
train_columns = train_split.column_names

sample_row = train_split[0]


def looks_like_image_value(value):
    """
    Return True when a value appears to represent an image.
    """

    if isinstance(value, PILImage.Image):
        return True

    if isinstance(value, np.ndarray):
        return value.ndim in (2, 3)

    if isinstance(value, Mapping):
        return (
            value.get("path") is not None
            or value.get("bytes") is not None
        )

    return False


# Respect an existing IMAGE_COLUMN variable when available.
existing_image_column = globals().get(
    "IMAGE_COLUMN"
)

if (
    existing_image_column is not None
    and existing_image_column in train_columns
):
    IMAGE_COLUMN = existing_image_column
else:
    image_feature_columns = [
        column_name
        for column_name, feature in train_features.items()
        if feature.__class__.__name__ == "Image"
    ]

    if len(image_feature_columns) == 1:
        IMAGE_COLUMN = image_feature_columns[0]
    else:
        image_name_candidates = [
            "image",
            "images",
            "img",
            "pixel_values",
            "image_path",
            "file_path",
            "filepath",
        ]

        IMAGE_COLUMN = next(
            (
                column_name
                for column_name in image_name_candidates
                if column_name in train_columns
            ),
            None,
        )

        if IMAGE_COLUMN is None:
            detected_image_columns = [
                column_name
                for column_name, value in sample_row.items()
                if looks_like_image_value(value)
            ]

            if len(detected_image_columns) != 1:
                raise ValueError(
                    "Unable to identify one unambiguous image "
                    "column. Found possible columns: "
                    f"{detected_image_columns}"
                )

            IMAGE_COLUMN = detected_image_columns[0]


# Respect an existing LABEL_COLUMN variable when available.
existing_label_column = globals().get(
    "LABEL_COLUMN"
)

if (
    existing_label_column is not None
    and existing_label_column in train_columns
):
    LABEL_COLUMN = existing_label_column
else:
    class_label_columns = [
        column_name
        for column_name, feature in train_features.items()
        if feature.__class__.__name__ == "ClassLabel"
    ]

    if len(class_label_columns) == 1:
        LABEL_COLUMN = class_label_columns[0]
    else:
        label_name_candidates = [
            "label",
            "labels",
            "class_id",
            "class",
            "target",
        ]

        LABEL_COLUMN = next(
            (
                column_name
                for column_name in label_name_candidates
                if column_name in train_columns
            ),
            None,
        )

        if LABEL_COLUMN is None:
            raise ValueError(
                "Unable to identify the label column. "
                f"Available columns: {train_columns}"
            )


if IMAGE_COLUMN == LABEL_COLUMN:
    raise ValueError(
        "The detected image and label columns are identical."
    )



# Normalize LABEL_TO_CLASS to an integer-ID-to-name mapping

def is_integer_like(value):
    if isinstance(value, (int, np.integer)):
        return True

    if isinstance(value, str):
        return value.strip().lstrip("-").isdigit()

    return False


if isinstance(LABEL_TO_CLASS, (list, tuple)):
    ID_TO_CLASS = {
        index: str(class_name)
        for index, class_name in enumerate(
            LABEL_TO_CLASS
        )
    }

elif isinstance(LABEL_TO_CLASS, Mapping):
    mapping_keys = list(LABEL_TO_CLASS.keys())
    mapping_values = list(LABEL_TO_CLASS.values())

    if all(
        is_integer_like(key)
        for key in mapping_keys
    ):
        ID_TO_CLASS = {
            int(label_id): str(class_name)
            for label_id, class_name
            in LABEL_TO_CLASS.items()
        }

    elif all(
        is_integer_like(value)
        for value in mapping_values
    ):
        ID_TO_CLASS = {
            int(label_id): str(class_name)
            for class_name, label_id
            in LABEL_TO_CLASS.items()
        }

    else:
        raise ValueError(
            "LABEL_TO_CLASS must map integer IDs to class "
            "names or class names to integer IDs."
        )

else:
    raise TypeError(
        "LABEL_TO_CLASS must be a dictionary, list or tuple."
    )


ID_TO_CLASS = dict(
    sorted(ID_TO_CLASS.items())
)

expected_label_ids = set(
    range(len(ID_TO_CLASS))
)

if set(ID_TO_CLASS.keys()) != expected_label_ids:
    raise ValueError(
        "The class IDs must be consecutive integers beginning "
        f"at zero. Found: {sorted(ID_TO_CLASS.keys())}"
    )


NUM_CLASSES = len(ID_TO_CLASS)
CLASS_NAMES = [
    ID_TO_CLASS[label_id]
    for label_id in range(NUM_CLASSES)
]

CLASS_TO_LABEL = {
    class_name: label_id
    for label_id, class_name in ID_TO_CLASS.items()
}


if NUM_CLASSES != 4:
    raise ValueError(
        "MaizeGuard should contain exactly four classes, "
        f"but {NUM_CLASSES} classes were found."
    )


print("Image column:", IMAGE_COLUMN)
print("Label column:", LABEL_COLUMN)
print("Physical-leaf column:", GROUP_COLUMN)
print("Number of classes:", NUM_CLASSES)

print("\nClass mapping:")
for label_id, class_name in ID_TO_CLASS.items():
    print(f"  {label_id}: {class_name}")

In [ ]:
# Define robust image decoding and label conversion functions


def normalize_class_text(value):
    """
    Normalize a class name for reliable string matching.
    """

    return re.sub(
        r"[^a-z0-9]+",
        "",
        str(value).strip().lower(),
    )


NORMALIZED_CLASS_TO_LABEL = {
    normalize_class_text(class_name): label_id
    for class_name, label_id
    in CLASS_TO_LABEL.items()
}


def encode_label(label_value):
    """
    Convert an integer or string label to an integer class ID.
    """

    if tf.is_tensor(label_value):
        label_value = label_value.numpy()

    if isinstance(label_value, np.ndarray):
        if label_value.size != 1:
            raise ValueError(
                "A label must contain exactly one value."
            )
        label_value = label_value.item()

    if isinstance(label_value, np.generic):
        label_value = label_value.item()

    if isinstance(label_value, bytes):
        label_value = label_value.decode("utf-8")

    if isinstance(
        label_value,
        (int, np.integer),
    ):
        encoded_label = int(label_value)

    elif isinstance(
        label_value,
        (float, np.floating),
    ):
        if not float(label_value).is_integer():
            raise ValueError(
                f"Non-integer numeric label: {label_value}"
            )

        encoded_label = int(label_value)

    elif isinstance(label_value, str):
        stripped_label = label_value.strip()

        if stripped_label.lstrip("-").isdigit():
            encoded_label = int(stripped_label)
        else:
            normalized_label = normalize_class_text(
                stripped_label
            )

            if (
                normalized_label
                not in NORMALIZED_CLASS_TO_LABEL
            ):
                raise ValueError(
                    "Unknown string label: "
                    f"'{label_value}'"
                )

            encoded_label = (
                NORMALIZED_CLASS_TO_LABEL[
                    normalized_label
                ]
            )

    else:
        raise TypeError(
            "Unsupported label type: "
            f"{type(label_value)}"
        )

    if encoded_label not in ID_TO_CLASS:
        raise ValueError(
            f"Label ID {encoded_label} is outside the "
            f"valid range 0 to {NUM_CLASSES - 1}."
        )

    return encoded_label


def image_to_rgb_array(image_value):
    """
    Convert a Hugging Face image value, PIL image, file path,
    encoded bytes or NumPy array into an H x W x 3 uint8 array.
    """

    image = None

    if isinstance(image_value, PILImage.Image):
        image = image_value

    elif isinstance(image_value, Mapping):
        image_bytes = image_value.get("bytes")
        image_path = image_value.get("path")

        if image_bytes is not None:
            image = PILImage.open(
                BytesIO(image_bytes)
            )

        elif image_path is not None:
            image = PILImage.open(image_path)

        else:
            raise ValueError(
                "The image dictionary contains neither "
                "'bytes' nor 'path'."
            )

    elif isinstance(image_value, (str, Path)):
        image = PILImage.open(image_value)

    elif tf.is_tensor(image_value):
        image_value = image_value.numpy()

    if image is not None:
        image = ImageOps.exif_transpose(image)
        image = image.convert("RGB")

        image_array = np.asarray(
            image,
            dtype=np.uint8,
        )

    else:
        image_array = np.asarray(image_value)

        if image_array.ndim == 2:
            image_array = np.stack(
                [image_array] * 3,
                axis=-1,
            )

        elif image_array.ndim == 3:
            # Convert possible channels-first arrays.
            if (
                image_array.shape[0] in (1, 3, 4)
                and image_array.shape[-1] not in (1, 3, 4)
            ):
                image_array = np.transpose(
                    image_array,
                    (1, 2, 0),
                )

            if image_array.shape[-1] == 1:
                image_array = np.repeat(
                    image_array,
                    repeats=3,
                    axis=-1,
                )

            elif image_array.shape[-1] == 4:
                image_array = np.asarray(
                    PILImage.fromarray(
                        image_array.astype(np.uint8)
                    ).convert("RGB"),
                    dtype=np.uint8,
                )

            elif image_array.shape[-1] != 3:
                raise ValueError(
                    "Expected 1, 3 or 4 image channels, "
                    f"but found shape {image_array.shape}."
                )

        else:
            raise ValueError(
                "Expected a two- or three-dimensional image, "
                f"but found shape {image_array.shape}."
            )

        if image_array.dtype != np.uint8:
            image_array = image_array.astype(
                np.float32
            )

            finite_values = image_array[
                np.isfinite(image_array)
            ]

            if finite_values.size == 0:
                raise ValueError(
                    "Image contains no finite pixel values."
                )

            if (
                finite_values.min() >= 0.0
                and finite_values.max() <= 1.0
            ):
                image_array = image_array * 255.0

            image_array = np.clip(
                image_array,
                0.0,
                255.0,
            ).astype(np.uint8)

    if image_array.ndim != 3:
        raise ValueError(
            f"Invalid decoded image shape: {image_array.shape}"
        )

    if image_array.shape[-1] != 3:
        raise ValueError(
            "Decoded images must contain exactly three RGB "
            f"channels. Found shape: {image_array.shape}"
        )

    return np.ascontiguousarray(image_array)

In [ ]:
# Create raw TensorFlow datasets from the existing Hugging Face splits


RAW_OUTPUT_SIGNATURE = (
    tf.TensorSpec(
        shape=(None, None, 3),
        dtype=tf.uint8,
        name="image",
    ),
    tf.TensorSpec(
        shape=(),
        dtype=tf.int32,
        name="label",
    ),
)


def make_raw_tf_dataset(hf_split):
    """
    Create a tf.data.Dataset that reads the existing
    Hugging Face split without changing that split.
    """

    def example_generator():
        for row in hf_split:
            image_array = image_to_rgb_array(
                row[IMAGE_COLUMN]
            )

            label_id = encode_label(
                row[LABEL_COLUMN]
            )

            yield (
                image_array,
                np.int32(label_id),
            )

    raw_dataset = tf.data.Dataset.from_generator(
        example_generator,
        output_signature=RAW_OUTPUT_SIGNATURE,
    )

    raw_dataset = raw_dataset.apply(
        tf.data.experimental.assert_cardinality(
            len(hf_split)
        )
    )

    return raw_dataset


raw_train_tf_dataset = make_raw_tf_dataset(
    maize_dataset["train"]
)

raw_validation_tf_dataset = make_raw_tf_dataset(
    maize_dataset[VALIDATION_SPLIT_NAME]
)

raw_test_tf_dataset = make_raw_tf_dataset(
    maize_dataset["test"]
)


print("Raw TensorFlow datasets created.")
print(
    "Train element specification:",
    raw_train_tf_dataset.element_spec,
)
print(
    "Validation element specification:",
    raw_validation_tf_dataset.element_spec,
)
print(
    "Test element specification:",
    raw_test_tf_dataset.element_spec,
)

In [ ]:
# Test one raw image before constructing the full pipeline


sample_image, sample_label = next(
    iter(raw_train_tf_dataset.take(1))
)

if sample_image.dtype != tf.uint8:
    raise TypeError(
        "Raw images should have dtype uint8."
    )

if sample_image.shape.rank != 3:
    raise ValueError(
        "A raw image should have three dimensions."
    )

if sample_image.shape[-1] != 3:
    raise ValueError(
        "A raw image should contain three RGB channels."
    )

if int(sample_label.numpy()) not in ID_TO_CLASS:
    raise ValueError(
        "The sample label is outside the class mapping."
    )


print("Raw sample image shape:", sample_image.shape)
print("Raw sample image dtype:", sample_image.dtype)
print("Raw sample label:", int(sample_label.numpy()))
print(
    "Raw sample class:",
    ID_TO_CLASS[int(sample_label.numpy())],
)
print(
    "Raw pixel range:",
    int(tf.reduce_min(sample_image).numpy()),
    "to",
    int(tf.reduce_max(sample_image).numpy()),
)

In [ ]:
# Define deterministic resizing and normalization

@tf.function
def resize_image_to_uint8(image, label):
    """
    Convert an image to RGB, resize while retaining its aspect
    ratio and return a fixed-size uint8 image.

    Any required padding is added symmetrically.
    """

    image = tf.ensure_shape(
        image,
        [None, None, 3],
    )

    # Convert uint8 [0, 255] to float32 [0, 1].
    image = tf.image.convert_image_dtype(
        image,
        dtype=tf.float32,
    )

    image = tf.image.resize_with_pad(
        image,
        target_height=IMAGE_HEIGHT,
        target_width=IMAGE_WIDTH,
        method=tf.image.ResizeMethod.BILINEAR,
        antialias=True,
    )

    image = tf.clip_by_value(
        image,
        0.0,
        1.0,
    )

    # Cache the resized version as uint8 to use less memory.
    image = tf.image.convert_image_dtype(
        image,
        dtype=tf.uint8,
        saturate=True,
    )

    image = tf.ensure_shape(
        image,
        [IMAGE_HEIGHT, IMAGE_WIDTH, 3],
    )

    label = tf.cast(label, tf.int32)

    return image, label


@tf.function
def normalize_image(image, label):
    """
    Normalize uint8 pixels from [0, 255] to float32 [0, 1].
    """

    image = tf.image.convert_image_dtype(
        image,
        dtype=tf.float32,
    )

    image = tf.ensure_shape(
        image,
        [IMAGE_HEIGHT, IMAGE_WIDTH, 3],
    )

    label = tf.cast(label, tf.int32)

    return image, label


TF_DATA_OPTIONS = tf.data.Options()
TF_DATA_OPTIONS.experimental_deterministic = True


print("Deterministic preprocessing functions created.")
print("Model input shape:", (IMAGE_HEIGHT, IMAGE_WIDTH, 3))
print("Model pixel range: [0.0, 1.0]")
print("Label format: sparse integer labels")

In [ ]:
# Define the training augmentation block


data_augmentation = tf.keras.Sequential(
    [
        tf.keras.layers.RandomFlip(
            mode="horizontal_and_vertical",
            seed=SEED,
            name="random_flip",
        ),

        tf.keras.layers.RandomRotation(
            factor=0.05,
            fill_mode="reflect",
            interpolation="bilinear",
            seed=SEED + 1,
            name="random_rotation",
        ),

        tf.keras.layers.RandomTranslation(
            height_factor=0.05,
            width_factor=0.05,
            fill_mode="reflect",
            interpolation="bilinear",
            seed=SEED + 2,
            name="random_translation",
        ),

        tf.keras.layers.RandomZoom(
            height_factor=(-0.10, 0.10),
            width_factor=(-0.10, 0.10),
            fill_mode="reflect",
            interpolation="bilinear",
            seed=SEED + 3,
            name="random_zoom",
        ),

        tf.keras.layers.RandomContrast(
            factor=0.10,
            seed=SEED + 4,
            name="random_contrast",
        ),

        # RandomContrast can produce values slightly outside
        # [0, 1]. This layer clips them back to that range.
        tf.keras.layers.ReLU(
            max_value=1.0,
            name="clip_augmented_pixels",
        ),
    ],
    name="maizeguard_data_augmentation",
)


data_augmentation.build(
    input_shape=(
        None,
        IMAGE_HEIGHT,
        IMAGE_WIDTH,
        3,
    )
)

data_augmentation.summary()

In [ ]:
# Resize and cache the deterministic datasets

def resize_and_cache_dataset(
    raw_dataset,
    split_name,
):
    """
    Apply deterministic resizing once and cache the result.
    """

    dataset = raw_dataset.map(
        resize_image_to_uint8,
        num_parallel_calls=AUTOTUNE,
        deterministic=True,
    )

    if CACHE_IN_MEMORY:
        dataset = dataset.cache()
    else:
        TF_CACHE_DIRECTORY.mkdir(
            parents=True,
            exist_ok=True,
        )

        cache_path = (
            TF_CACHE_DIRECTORY
            / (
                f"{split_name}_"
                f"{IMAGE_HEIGHT}x{IMAGE_WIDTH}_uint8"
            )
        )

        dataset = dataset.cache(
            str(cache_path)
        )

    dataset = dataset.with_options(
        TF_DATA_OPTIONS
    )

    return dataset


cached_train_tf_dataset = resize_and_cache_dataset(
    raw_train_tf_dataset,
    "train",
)

cached_validation_tf_dataset = (
    resize_and_cache_dataset(
        raw_validation_tf_dataset,
        VALIDATION_SPLIT_NAME,
    )
)

cached_test_tf_dataset = resize_and_cache_dataset(
    raw_test_tf_dataset,
    "test",
)


print("Deterministic resize-and-cache stages created.")
print("Cache in memory:", CACHE_IN_MEMORY)

if not CACHE_IN_MEMORY:
    print("Cache directory:", TF_CACHE_DIRECTORY)

In [ ]:
# Build the final TensorFlow training, validation and test pipelines

@tf.function
def augment_training_batch(images, labels):
    """
    Apply augmentation only to a training batch.
    """

    augmented_images = data_augmentation(
        images,
        training=True,
    )

    return augmented_images, labels


# Training pipeline

train_tf_dataset = (
    cached_train_tf_dataset

    # Shuffle only the training split.
    .shuffle(
        buffer_size=len(maize_dataset["train"]),
        seed=SEED,
        reshuffle_each_iteration=True,
    )

    .map(
        normalize_image,
        num_parallel_calls=AUTOTUNE,
        deterministic=True,
    )

    .batch(
        BATCH_SIZE,
        drop_remainder=False,
    )

    .map(
        augment_training_batch,
        num_parallel_calls=AUTOTUNE,
        deterministic=True,
    )

    .prefetch(AUTOTUNE)
)



# Unaugmented training pipeline
#
# Useful for:
# - inspecting original preprocessed images;
# - evaluating performance on the training split;
# - verifying labels and class distributions.


train_eval_tf_dataset = (
    cached_train_tf_dataset

    .map(
        normalize_image,
        num_parallel_calls=AUTOTUNE,
        deterministic=True,
    )

    .batch(
        BATCH_SIZE,
        drop_remainder=False,
    )

    .prefetch(AUTOTUNE)
)



# Validation pipeline


validation_tf_dataset = (
    cached_validation_tf_dataset

    .map(
        normalize_image,
        num_parallel_calls=AUTOTUNE,
        deterministic=True,
    )

    .batch(
        BATCH_SIZE,
        drop_remainder=False,
    )

    .prefetch(AUTOTUNE)
)


# Convenient shorter alias.
val_tf_dataset = validation_tf_dataset



# Test pipeline


test_tf_dataset = (
    cached_test_tf_dataset

    .map(
        normalize_image,
        num_parallel_calls=AUTOTUNE,
        deterministic=True,
    )

    .batch(
        BATCH_SIZE,
        drop_remainder=False,
    )

    .prefetch(AUTOTUNE)
)



# Pipeline metadata for Phase 4


STEPS_PER_EPOCH = math.ceil(
    len(maize_dataset["train"]) / BATCH_SIZE
)

VALIDATION_STEPS = math.ceil(
    len(
        maize_dataset[
            VALIDATION_SPLIT_NAME
        ]
    )
    / BATCH_SIZE
)

TEST_STEPS = math.ceil(
    len(maize_dataset["test"]) / BATCH_SIZE
)


PHASE_3_CONFIG = {
    "image_size": IMAGE_SIZE,
    "input_shape": (
        IMAGE_HEIGHT,
        IMAGE_WIDTH,
        3,
    ),
    "batch_size": BATCH_SIZE,
    "number_of_classes": NUM_CLASSES,
    "class_names": CLASS_NAMES,
    "steps_per_epoch": STEPS_PER_EPOCH,
    "validation_steps": VALIDATION_STEPS,
    "test_steps": TEST_STEPS,
    "pixel_range": (0.0, 1.0),
    "label_format": "sparse_integer",
    "augmentation_location": (
        "train_tf_dataset"
    ),
}


print("Final TensorFlow pipelines created.")
print("Training batches per epoch:", STEPS_PER_EPOCH)
print("Validation batches:", VALIDATION_STEPS)
print("Test batches:", TEST_STEPS)

In [ ]:
# Inspecting  pipeline shapes, data types and value ranges


def inspect_tf_pipeline(
    dataset,
    dataset_name,
):
    images, labels = next(
        iter(dataset.take(1))
    )

    image_minimum = float(
        tf.reduce_min(images).numpy()
    )

    image_maximum = float(
        tf.reduce_max(images).numpy()
    )

    label_minimum = int(
        tf.reduce_min(labels).numpy()
    )

    label_maximum = int(
        tf.reduce_max(labels).numpy()
    )

    if images.shape.rank != 4:
        raise ValueError(
            f"{dataset_name}: expected batched images "
            "with four dimensions."
        )

    if tuple(images.shape[1:]) != (
        IMAGE_HEIGHT,
        IMAGE_WIDTH,
        3,
    ):
        raise ValueError(
            f"{dataset_name}: unexpected image shape "
            f"{images.shape}."
        )

    if images.dtype != tf.float32:
        raise TypeError(
            f"{dataset_name}: expected float32 images, "
            f"but found {images.dtype}."
        )

    if labels.dtype != tf.int32:
        raise TypeError(
            f"{dataset_name}: expected int32 labels, "
            f"but found {labels.dtype}."
        )

    if image_minimum < 0.0 or image_maximum > 1.0:
        raise ValueError(
            f"{dataset_name}: pixels are outside [0, 1]."
        )

    if (
        label_minimum < 0
        or label_maximum >= NUM_CLASSES
    ):
        raise ValueError(
            f"{dataset_name}: labels are outside the "
            "valid class range."
        )

    if not bool(
        tf.reduce_all(
            tf.math.is_finite(images)
        ).numpy()
    ):
        raise ValueError(
            f"{dataset_name}: non-finite pixels detected."
        )

    print(f"\n{dataset_name}")
    print("  Image shape:", images.shape)
    print("  Image dtype:", images.dtype)
    print(
        "  Pixel range:",
        f"{image_minimum:.4f}",
        "to",
        f"{image_maximum:.4f}",
    )
    print("  Label shape:", labels.shape)
    print("  Label dtype:", labels.dtype)
    print(
        "  Label range:",
        label_minimum,
        "to",
        label_maximum,
    )


inspect_tf_pipeline(
    train_tf_dataset,
    "Augmented training pipeline",
)

inspect_tf_pipeline(
    train_eval_tf_dataset,
    "Unaugmented training pipeline",
)

inspect_tf_pipeline(
    validation_tf_dataset,
    "Validation pipeline",
)

inspect_tf_pipeline(
    test_tf_dataset,
    "Test pipeline",
)

In [ ]:
# Visualizing original and augmented training images



original_images, original_labels = next(
    iter(train_eval_tf_dataset.take(1))
)

augmented_images = data_augmentation(
    original_images,
    training=True,
)

number_to_display = min(
    6,
    int(original_images.shape[0]),
)

plt.figure(figsize=(15, 7))

for image_index in range(number_to_display):
    label_id = int(
        original_labels[image_index].numpy()
    )

    class_name = ID_TO_CLASS[label_id]

    # Original preprocessed image.
    plt.subplot(
        2,
        number_to_display,
        image_index + 1,
    )

    plt.imshow(
        original_images[image_index]
    )

    plt.title(
        f"Original\n{class_name}",
        fontsize=9,
    )

    plt.axis("off")

    # Augmented version.
    plt.subplot(
        2,
        number_to_display,
        number_to_display + image_index + 1,
    )

    plt.imshow(
        augmented_images[image_index]
    )

    plt.title(
        f"Augmented\n{class_name}",
        fontsize=9,
    )

    plt.axis("off")


plt.suptitle(
    "MaizeGuard Training Augmentation Check",
    fontsize=14,
)

plt.tight_layout()
plt.show()

In [ ]:
# Verifying that all images and labels pass through the pipelines


def audit_complete_tf_pipeline(
    dataset,
    dataset_name,
):
    """
    Count every example and verify that all batches contain
    finite normalized images and valid labels.
    """

    label_counts = np.zeros(
        NUM_CLASSES,
        dtype=np.int64,
    )

    processed_examples = 0
    global_minimum = np.inf
    global_maximum = -np.inf

    for images, labels in dataset:
        if not bool(
            tf.reduce_all(
                tf.math.is_finite(images)
            ).numpy()
        ):
            raise ValueError(
                f"{dataset_name} contains non-finite pixels."
            )

        batch_minimum = float(
            tf.reduce_min(images).numpy()
        )

        batch_maximum = float(
            tf.reduce_max(images).numpy()
        )

        global_minimum = min(
            global_minimum,
            batch_minimum,
        )

        global_maximum = max(
            global_maximum,
            batch_maximum,
        )

        label_array = labels.numpy().astype(
            np.int64
        )

        if np.any(label_array < 0):
            raise ValueError(
                f"{dataset_name} contains negative labels."
            )

        if np.any(label_array >= NUM_CLASSES):
            raise ValueError(
                f"{dataset_name} contains labels outside "
                "the valid class range."
            )

        label_counts += np.bincount(
            label_array,
            minlength=NUM_CLASSES,
        )

        processed_examples += len(
            label_array
        )

    if global_minimum < 0.0:
        raise ValueError(
            f"{dataset_name} contains pixels below zero."
        )

    if global_maximum > 1.0:
        raise ValueError(
            f"{dataset_name} contains pixels above one."
        )

    return {
        "examples": processed_examples,
        "label_counts": label_counts,
        "minimum_pixel": global_minimum,
        "maximum_pixel": global_maximum,
    }


pipeline_audit_results = {
    "train": audit_complete_tf_pipeline(
        train_eval_tf_dataset,
        "Training pipeline",
    ),

    VALIDATION_SPLIT_NAME:
        audit_complete_tf_pipeline(
            validation_tf_dataset,
            "Validation pipeline",
        ),

    "test": audit_complete_tf_pipeline(
        test_tf_dataset,
        "Test pipeline",
    ),
}


for split_name, expected_size in (
    EXPECTED_SPLIT_SIZES.items()
):
    observed_size = (
        pipeline_audit_results[
            split_name
        ]["examples"]
    )

    if observed_size != expected_size:
        raise ValueError(
            f"{split_name}: expected {expected_size:,} "
            f"examples, but the TensorFlow pipeline "
            f"produced {observed_size:,}."
        )


print("Full TensorFlow pipeline audit passed.")

for split_name, results in (
    pipeline_audit_results.items()
):
    print(f"\n{split_name.upper()}")

    print(
        "  Processed examples:",
        f"{results['examples']:,}",
    )

    print(
        "  Pixel range:",
        f"{results['minimum_pixel']:.4f}",
        "to",
        f"{results['maximum_pixel']:.4f}",
    )

    print("  Class counts:")

    for label_id, class_count in enumerate(
        results["label_counts"]
    ):
        print(
            f"    {label_id} - "
            f"{ID_TO_CLASS[label_id]}: "
            f"{class_count:,}"
        )

In [ ]:
# Final Phase 3 completion check


phase_3_pipeline_total = sum(
    results["examples"]
    for results in pipeline_audit_results.values()
)

if phase_3_pipeline_total != 3852:
    raise ValueError(
        "Phase 3 pipeline total does not equal 3,852."
    )


print("=" * 64)
print("PHASE 3 COMPLETED SUCCESSFULLY")
print("=" * 64)

print(
    f"Input image size: "
    f"{IMAGE_HEIGHT} x {IMAGE_WIDTH} x 3"
)

print("Input dtype: float32")
print("Input pixel range: [0, 1]")
print("Label format: sparse integer labels")
print(f"Number of classes: {NUM_CLASSES}")
print(f"Batch size: {BATCH_SIZE}")

print("\nPrepared TensorFlow datasets:")
print(
    "  train_tf_dataset       "
    "- shuffled and augmented"
)
print(
    "  train_eval_tf_dataset  "
    "- unaugmented training data"
)
print(
    "  validation_tf_dataset  "
    "- deterministic validation data"
)
print(
    "  val_tf_dataset         "
    "- validation alias"
)
print(
    "  test_tf_dataset        "
    "- deterministic test data"
)

print(
    "\nTotal images passing through pipelines:",
    f"{phase_3_pipeline_total:,}",
)

print(
    "Original Hugging Face DatasetDict retained as:",
    "maize_dataset",
)

print(
    "\nImportant for Phase 4:"
    "\nThe augmentation block is already applied inside "
    "train_tf_dataset. Do not add data_augmentation to the "
    "model again, because that would augment each training "
    "image twice."
)


PHASE 4 — CLASSICAL MACHINE-LEARNING BASELINES


 Goal:
 Build classical ML baselines using handcrafted image features
 before moving to deep learning.

 Models:
 - Logistic Regression
 - RBF Support Vector Machine
 - Random Forest

Model selection:
- Train using TRAIN split
- Select best model using VALIDATION macro-F1
- Evaluate selected model once on TEST split

In [ ]:

# IMPORTS


from tqdm.auto import tqdm

from skimage.color import rgb2gray
from skimage.feature import hog
from skimage.transform import resize

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (
    accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix
)

import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt



# FEATURE EXTRACTION SETTINGS


FEATURE_IMAGE_SIZE = (128, 128)

HIST_BINS = 16



# HANDCRAFTED FEATURE EXTRACTION


def extract_handcrafted_features(pil_image):
    """
    Extract HOG texture features and normalized RGB
    histogram features from one maize leaf image.
    """

    # Convert image to RGB and scale pixels to [0, 1]
    image = np.asarray(
        pil_image.convert("RGB"),
        dtype=np.float32
    ) / 255.0


    # Resize image
    image = resize(
        image,
        FEATURE_IMAGE_SIZE,
        anti_aliasing=True,
        preserve_range=True,
    ).astype(np.float32)



    # HOG FEATURES


    gray = rgb2gray(image)

    hog_features = hog(
        gray,
        orientations=9,
        pixels_per_cell=(16, 16),
        cells_per_block=(2, 2),
        block_norm="L2-Hys",
        feature_vector=True,
    ).astype(np.float32)



    # RGB HISTOGRAM FEATURES


    histogram_features = []

    for channel in range(3):

        hist, _ = np.histogram(
            image[:, :, channel],
            bins=HIST_BINS,
            range=(0.0, 1.0),
            density=False
        )

        hist = hist.astype(np.float32)

        # Normalize histogram
        hist /= hist.sum() + 1e-8

        histogram_features.extend(hist)



    # COMBINE FEATURES


    features = np.concatenate([
        hog_features,
        np.asarray(
            histogram_features,
            dtype=np.float32
        )
    ])

    return features



# BUILD FEATURE MATRIX


def build_feature_matrix(hf_split, split_name):
    """
    Convert a Hugging Face dataset split into:
        X = feature matrix
        y = class labels
    """

    X = []
    y = []

    for row in tqdm(
        hf_split,
        total=len(hf_split),
        desc=f"Extracting features: {split_name}"
    ):

        features = extract_handcrafted_features(
            row["image"]
        )

        X.append(features)

        y.append(
            int(row["label"])
        )


    return (
        np.asarray(
            X,
            dtype=np.float32
        ),
        np.asarray(
            y,
            dtype=np.int64
        )
    )



# EXTRACT FEATURES


print("=" * 70)

print(
    "PHASE 4 — CLASSICAL MACHINE-LEARNING BASELINES"
)

print("=" * 70)

print(
    "\nExtracting handcrafted image features..."
)


X_train, y_train = build_feature_matrix(
    maize_dataset["train"],
    "train"
)


X_val, y_val = build_feature_matrix(
    maize_dataset["validation"],
    "validation"
)


X_test, y_test = build_feature_matrix(
    maize_dataset["test"],
    "test"
)


print(
    "\nFeature extraction complete."
)


print(
    "\nTrain:",
    X_train.shape,
    y_train.shape
)

print(
    "Validation:",
    X_val.shape,
    y_val.shape
)

print(
    "Test:",
    X_test.shape,
    y_test.shape
)



# CHECK CLASS DISTRIBUTION


print(
    "\nClass distribution:"
)


for split_name, labels in [
    ("Train", y_train),
    ("Validation", y_val),
    ("Test", y_test)
]:

    unique_classes, counts = np.unique(
        labels,
        return_counts=True
    )

    print(
        f"\n{split_name}:"
    )

    for class_id, count in zip(
        unique_classes,
        counts
    ):

        print(
            f"  {class_id} "
            f"({ID_TO_CLASS[int(class_id)]}): "
            f"{count}"
        )



# DEFINE BASELINE MODELS


baseline_models = {



    # LOGISTIC REGRESSION


    "logistic_regression":

        Pipeline([

            (
                "scaler",
                StandardScaler()
            ),

            (
                "model",

                LogisticRegression(
                    max_iter=2500,
                    class_weight="balanced",
                    random_state=SEED
                )
            )

        ]),



    # RBF SUPPORT VECTOR MACHINE


    "rbf_svm":

        Pipeline([

            (
                "scaler",
                StandardScaler()
            ),

            (
                "model",

                SVC(
                    kernel="rbf",
                    C=5.0,
                    gamma="scale",
                    probability=True,
                    class_weight="balanced",
                    random_state=SEED
                )
            )

        ]),



    # RANDOM FOREST


    "random_forest":

        RandomForestClassifier(

            n_estimators=400,

            class_weight=
                "balanced_subsample",

            random_state=SEED,

            n_jobs=-1
        )
}



# TRAIN BASELINE MODELS


baseline_results = []

trained_baselines = {}


for model_name, model in baseline_models.items():


    print(
        "\n" + "=" * 70
    )

    print(
        f"TRAINING MODEL: {model_name}"
    )

    print(
        "=" * 70
    )



    # TRAIN


    model.fit(
        X_train,
        y_train
    )


    trained_baselines[
        model_name
    ] = model



    # VALIDATION PREDICTIONS


    val_pred = model.predict(
        X_val
    )



    # VALIDATION METRICS


    validation_accuracy = (
        accuracy_score(
            y_val,
            val_pred
        )
    )


    validation_macro_f1 = (
        f1_score(
            y_val,
            val_pred,
            average="macro"
        )
    )


    validation_weighted_f1 = (
        f1_score(
            y_val,
            val_pred,
            average="weighted"
        )
    )


    result = {

        "model":
            model_name,

        "validation_accuracy":
            validation_accuracy,

        "validation_macro_f1":
            validation_macro_f1,

        "validation_weighted_f1":
            validation_weighted_f1
    }


    baseline_results.append(
        result
    )


    print(
        f"\nValidation accuracy: "
        f"{validation_accuracy:.4f}"
    )


    print(
        f"Validation macro-F1: "
        f"{validation_macro_f1:.4f}"
    )


    print(
        f"Validation weighted-F1: "
        f"{validation_weighted_f1:.4f}"
    )



# COMPARE MODELS


baseline_results_df = (

    pd.DataFrame(
        baseline_results
    )

    .sort_values(
        "validation_macro_f1",
        ascending=False
    )

    .reset_index(
        drop=True
    )
)


print(
    "\n" + "=" * 70
)

print(
    "BASELINE MODEL COMPARISON"
)

print(
    "=" * 70
)


display(
    baseline_results_df
)



# SELECT BEST MODEL


best_baseline_name = (
    baseline_results_df.loc[
        0,
        "model"
    ]
)


best_baseline = (
    trained_baselines[
        best_baseline_name
    ]
)


best_validation_macro_f1 = (

    baseline_results_df.loc[
        0,
        "validation_macro_f1"
    ]
)


print(
    "\n" + "=" * 70
)

print(
    "SELECTED CLASSICAL BASELINE"
)

print(
    "=" * 70
)


print(
    f"\nBest model: "
    f"{best_baseline_name}"
)


print(
    f"Validation macro-F1: "
    f"{best_validation_macro_f1:.4f}"
)



# SAVE BEST MODEL


BASELINE_MODEL_PATH = (

    MODEL_DIR /
    "phase4_best_classical_baseline.joblib"
)


BASELINE_RESULTS_PATH = (

    OUTPUT_DIR /
    "phase4_baseline_validation_results.csv"
)


joblib.dump(
    best_baseline,
    BASELINE_MODEL_PATH
)


baseline_results_df.to_csv(
    BASELINE_RESULTS_PATH,
    index=False
)


print(
    "\nSaved best baseline model:"
)

print(
    BASELINE_MODEL_PATH
)


print(
    "\nSaved validation comparison:"
)

print(
    BASELINE_RESULTS_PATH
)



# FINAL TEST EVALUATION


print(
    "\n" + "=" * 70
)

print(
    "FINAL TEST EVALUATION"
)

print(
    "=" * 70
)


# The test set is evaluated only after
# model selection using validation data.

baseline_test_pred = (
    best_baseline.predict(
        X_test
    )
)



# TEST METRICS


test_accuracy = (
    accuracy_score(
        y_test,
        baseline_test_pred
    )
)


test_macro_f1 = (
    f1_score(
        y_test,
        baseline_test_pred,
        average="macro"
    )
)


test_weighted_f1 = (
    f1_score(
        y_test,
        baseline_test_pred,
        average="weighted"
    )
)


baseline_test_metrics = {

    "model":
        best_baseline_name,

    "test_accuracy":
        test_accuracy,

    "test_macro_f1":
        test_macro_f1,

    "test_weighted_f1":
        test_weighted_f1
}


baseline_test_metrics_df = (

    pd.DataFrame([
        baseline_test_metrics
    ])
)


print(
    "\nTest performance:"
)


display(
    baseline_test_metrics_df
)



# CLASSIFICATION REPORT


print(
    "\n" + "=" * 70
)

print(
    "TEST CLASSIFICATION REPORT"
)

print(
    "=" * 70
)


print(

    classification_report(

        y_test,

        baseline_test_pred,

        labels=list(
            range(NUM_CLASSES)
        ),

        target_names=
            CLASS_NAMES,

        digits=4,

        zero_division=0
    )
)



# CONFUSION MATRIX


cm = confusion_matrix(
    y_test,
    baseline_test_pred,
    labels=list(
        range(NUM_CLASSES)
    )
)


print(
    "\nConfusion Matrix:"
)

print(
    cm
)



# PLOT CONFUSION MATRIX


fig, ax = plt.subplots(
    figsize=(8, 6)
)


im = ax.imshow(
    cm
)


ax.set_xticks(
    np.arange(
        NUM_CLASSES
    )
)


ax.set_yticks(
    np.arange(
        NUM_CLASSES
    )
)


ax.set_xticklabels(
    CLASS_NAMES,
    rotation=45,
    ha="right"
)


ax.set_yticklabels(
    CLASS_NAMES
)


ax.set_xlabel(
    "Predicted label"
)


ax.set_ylabel(
    "True label"
)


ax.set_title(
    f"Phase 4 Confusion Matrix — "
    f"{best_baseline_name}"
)


# Add values inside matrix cells

for i in range(
    NUM_CLASSES
):

    for j in range(
        NUM_CLASSES
    ):

        ax.text(
            j,
            i,
            cm[i, j],
            ha="center",
            va="center"
        )


plt.tight_layout()

plt.show()



# SAVE TEST METRICS


BASELINE_TEST_RESULTS_PATH = (

    OUTPUT_DIR /
    "phase4_best_baseline_test_metrics.csv"
)


baseline_test_metrics_df.to_csv(

    BASELINE_TEST_RESULTS_PATH,

    index=False
)



# SAVE CLASSIFICATION REPORT


classification_report_dict = (

    classification_report(

        y_test,

        baseline_test_pred,

        labels=list(
            range(NUM_CLASSES)
        ),

        target_names=
            CLASS_NAMES,

        output_dict=True,

        zero_division=0
    )
)


classification_report_df = (

    pd.DataFrame(
        classification_report_dict
    ).transpose()
)


CLASSIFICATION_REPORT_PATH = (

    OUTPUT_DIR /
    "phase4_classification_report.csv"
)


classification_report_df.to_csv(

    CLASSIFICATION_REPORT_PATH,

    index=True
)



# FINAL SUMMARY


print(
    "\n" + "=" * 70
)

print(
    "PHASE 4 SUMMARY"
)

print(
    "=" * 70
)


print(
    f"\nSelected baseline model: "
    f"{best_baseline_name}"
)


print(
    f"Validation macro-F1: "
    f"{best_validation_macro_f1:.4f}"
)


print(
    f"Test accuracy: "
    f"{test_accuracy:.4f}"
)


print(
    f"Test macro-F1: "
    f"{test_macro_f1:.4f}"
)


print(
    f"Test weighted-F1: "
    f"{test_weighted_f1:.4f}"
)


print(
    "\nSaved files:"
)


print(
    f"1. Model: "
    f"{BASELINE_MODEL_PATH}"
)


print(
    f"2. Validation results: "
    f"{BASELINE_RESULTS_PATH}"
)


print(
    f"3. Test metrics: "
    f"{BASELINE_TEST_RESULTS_PATH}"
)


print(
    f"4. Classification report: "
    f"{CLASSIFICATION_REPORT_PATH}"
)


print(
    "\n" + "=" * 70
)

print(
    "PHASE 4 COMPLETE"
)

print(
    "=" * 70
)

Phase 5 develops the MaizeGuard deep-learning model using EfficientNetB0 with transfer learning. The model is trained in two stages—first with the pretrained backbone frozen, then with selected upper layers fine-tuned—and the best checkpoint is chosen using validation performance.

In [ ]:
# PHASE 5 — DEEP-LEARNING MODEL DEVELOPMENT

import tensorflow as tf
import numpy as np
import pandas as pd

from tensorflow.keras import layers
from sklearn.utils.class_weight import compute_class_weight

print("TensorFlow version:", tf.__version__)

IMG_SIZE = (224, 224)
BATCH_SIZE = 32
AUTOTUNE = tf.data.AUTOTUNE


def hf_generator(hf_split):
    for row in hf_split:
        image = np.asarray(
            row["image"].convert("RGB"),
            dtype=np.uint8
        )

        label = np.int64(
            row["label"]
        )

        yield image, label


def make_tf_dataset(hf_split, training=False):

    ds = tf.data.Dataset.from_generator(
        lambda: hf_generator(hf_split),

        output_signature=(
            tf.TensorSpec(
                shape=(None, None, 3),
                dtype=tf.uint8
            ),

            tf.TensorSpec(
                shape=(),
                dtype=tf.int64
            ),
        ),
    )

    def preprocess(image, label):

        image = tf.image.resize(
            image,
            IMG_SIZE,
            antialias=True
        )

        image = tf.cast(
            image,
            tf.float32
        )

        return image, label

    ds = ds.map(
        preprocess,
        num_parallel_calls=AUTOTUNE
    )

    if training:

        ds = ds.shuffle(
            buffer_size=min(
                len(hf_split),
                2048
            ),

            seed=SEED,

            reshuffle_each_iteration=True
        )

    ds = ds.batch(
        BATCH_SIZE
    )

    ds = ds.prefetch(
        AUTOTUNE
    )

    return ds


train_tf = make_tf_dataset(
    maize_dataset["train"],
    training=True
)

val_tf = make_tf_dataset(
    maize_dataset["validation"],
    training=False
)

test_tf = make_tf_dataset(
    maize_dataset["test"],
    training=False
)


images, labels = next(
    iter(train_tf)
)

print("\nInput pipeline check:")

print(
    "Batch images:",
    images.shape,
    images.dtype
)

print(
    "Pixel range:",
    float(tf.reduce_min(images)),
    "to",
    float(tf.reduce_max(images))
)

print(
    "Batch labels:",
    labels.shape
)


augmentation = tf.keras.Sequential(
    [
        tf.keras.layers.RandomFlip(
            "horizontal"
        ),

        tf.keras.layers.RandomRotation(
            0.10
        ),

        tf.keras.layers.RandomZoom(
            0.10
        ),

        tf.keras.layers.RandomContrast(
            0.10
        ),
    ],

    name="data_augmentation"
)


train_labels = np.asarray(
    maize_dataset["train"]["label"],
    dtype=np.int64
)


weights = compute_class_weight(
    class_weight="balanced",

    classes=np.arange(
        NUM_CLASSES
    ),

    y=train_labels
)


CLASS_WEIGHTS = {
    i: float(w)
    for i, w in enumerate(weights)
}


print(
    "\nClass weights:",
    CLASS_WEIGHTS
)


tf.keras.backend.clear_session()

tf.keras.utils.set_random_seed(
    SEED
)


inputs = tf.keras.Input(
    shape=(*IMG_SIZE, 3),
    name="image"
)


x = augmentation(
    inputs
)


backbone = tf.keras.applications.EfficientNetB0(
    include_top=False,

    weights="imagenet",

    input_shape=(
        *IMG_SIZE,
        3
    )
)


backbone.trainable = False


x = backbone(
    x,
    training=False
)


x = layers.GlobalAveragePooling2D(
    name="global_average_pooling"
)(x)


x = layers.Dropout(
    0.35,
    name="dropout"
)(x)


outputs = layers.Dense(
    NUM_CLASSES,

    activation="softmax",

    name="predictions"
)(x)


model = tf.keras.Model(
    inputs,
    outputs,

    name="maizeguard_efficientnetb0"
)


model.compile(

    optimizer=tf.keras.optimizers.Adam(
        learning_rate=1e-3
    ),

    loss=tf.keras.losses.SparseCategoricalCrossentropy(),

    metrics=[
        tf.keras.metrics.SparseCategoricalAccuracy(
            name="accuracy"
        )
    ]
)


print("\nModel architecture:")

model.summary()


HEAD_MODEL_PATH = (
    MODEL_DIR /
    "phase5_efficientnet_head.keras"
)


head_callbacks = [

    tf.keras.callbacks.ModelCheckpoint(
        filepath=str(
            HEAD_MODEL_PATH
        ),

        monitor="val_loss",

        mode="min",

        save_best_only=True,

        verbose=1
    ),

    tf.keras.callbacks.EarlyStopping(
        monitor="val_loss",

        mode="min",

        patience=5,

        restore_best_weights=True,

        verbose=1
    ),

    tf.keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss",

        mode="min",

        factor=0.3,

        patience=2,

        min_lr=1e-6,

        verbose=1
    )
]


print(
    "\nTraining EfficientNetB0 classifier head..."
)


history_head = model.fit(

    train_tf,

    validation_data=val_tf,

    epochs=15,

    class_weight=CLASS_WEIGHTS,

    callbacks=head_callbacks
)


head_history_df = pd.DataFrame(
    history_head.history
)


head_history_df.to_csv(
    OUTPUT_DIR /
    "phase5_head_training_history.csv",

    index=False
)


print(
    "\nClassifier-head training complete."
)


model = tf.keras.models.load_model(
    HEAD_MODEL_PATH
)


backbone = model.get_layer(
    "efficientnetb0"
)


backbone.trainable = True


for layer in backbone.layers[:-30]:
    layer.trainable = False


for layer in backbone.layers:

    if isinstance(
        layer,
        tf.keras.layers.BatchNormalization
    ):

        layer.trainable = False


trainable_layers = sum(
    layer.trainable
    for layer in backbone.layers
)


print(
    "\nFine-tuning stage:"
)

print(
    "Total backbone layers:",
    len(backbone.layers)
)

print(
    "Trainable backbone layers:",
    trainable_layers
)


model.compile(

    optimizer=tf.keras.optimizers.Adam(
        learning_rate=1e-5
    ),

    loss=tf.keras.losses.SparseCategoricalCrossentropy(),

    metrics=[
        tf.keras.metrics.SparseCategoricalAccuracy(
            name="accuracy"
        )
    ]
)


FT_MODEL_PATH = (
    MODEL_DIR /
    "phase5_efficientnet_finetuned.keras"
)


finetune_callbacks = [

    tf.keras.callbacks.ModelCheckpoint(
        filepath=str(
            FT_MODEL_PATH
        ),

        monitor="val_loss",

        mode="min",

        save_best_only=True,

        verbose=1
    ),

    tf.keras.callbacks.EarlyStopping(
        monitor="val_loss",

        mode="min",

        patience=5,

        restore_best_weights=True,

        verbose=1
    ),

    tf.keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss",

        mode="min",

        factor=0.3,

        patience=2,

        min_lr=1e-7,

        verbose=1
    )
]


print(
    "\nFine-tuning EfficientNetB0..."
)


history_ft = model.fit(

    train_tf,

    validation_data=val_tf,

    epochs=15,

    class_weight=CLASS_WEIGHTS,

    callbacks=finetune_callbacks
)


finetune_history_df = pd.DataFrame(
    history_ft.history
)


finetune_history_df.to_csv(
    OUTPUT_DIR /
    "phase5_finetune_history.csv",

    index=False
)


print(
    "\nFine-tuning complete."
)


head_model = tf.keras.models.load_model(
    HEAD_MODEL_PATH
)


ft_model = tf.keras.models.load_model(
    FT_MODEL_PATH
)


head_val = head_model.evaluate(
    val_tf,

    verbose=0,

    return_dict=True
)


ft_val = ft_model.evaluate(
    val_tf,

    verbose=0,

    return_dict=True
)


comparison = pd.DataFrame(
    [
        {
            "checkpoint":
                "frozen_backbone",

            **head_val
        },

        {
            "checkpoint":
                "fine_tuned",

            **ft_val
        }
    ]
).sort_values(
    "loss"
).reset_index(
    drop=True
)


print(
    "\nCheckpoint comparison:"
)


display(
    comparison
)


if ft_val["loss"] <= head_val["loss"]:

    best_dl_model = ft_model

    selected_checkpoint = (
        "fine_tuned"
    )

else:

    best_dl_model = head_model

    selected_checkpoint = (
        "frozen_backbone"
    )


BEST_DL_MODEL_PATH = (
    MODEL_DIR /
    "maizeguard_best_efficientnetb0.keras"
)


best_dl_model.save(
    BEST_DL_MODEL_PATH
)


comparison.to_csv(
    OUTPUT_DIR /
    "phase5_checkpoint_comparison.csv",

    index=False
)


print(
    "\nSelected checkpoint:",
    selected_checkpoint
)


print(
    "Best validation loss:",
    comparison.loc[0, "loss"]
)


print(
    "Best validation accuracy:",
    comparison.loc[0, "accuracy"]
)


print(
    "\nSaved best model to:",
    BEST_DL_MODEL_PATH
)


print(
    "\nPHASE 5 COMPLETE"
)

Phase 6 is Model Evaluation and Error Analysis. It evaluates the final EfficientNetB0 model selected in Phase 5 on the untouched test set, then produces the classification metrics, confusion matrix, and misclassification analysis.

In [ ]:

# PHASE 6 — MODEL EVALUATION AND ERROR ANALYSIS


from sklearn.metrics import (
    accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix,
)


# Load the best deep-learning model selected in Phase 5


BEST_DL_MODEL_PATH = MODEL_DIR / "maizeguard_best_efficientnetb0.keras"

if not BEST_DL_MODEL_PATH.exists():
    raise FileNotFoundError(
        f"Best model not found at {BEST_DL_MODEL_PATH}. "
        "Run Phase 5 first."
    )

model = tf.keras.models.load_model(BEST_DL_MODEL_PATH)

print("Loaded model:", BEST_DL_MODEL_PATH)



# Generate predictions on the untouched test set


print("\nGenerating predictions on test set...")

test_probabilities = model.predict(
    test_tf,
    verbose=1
)

test_predictions = np.argmax(
    test_probabilities,
    axis=1
)

test_true = np.asarray(
    maize_dataset["test"]["label"],
    dtype=np.int64
)

assert len(test_predictions) == len(test_true), (
    "Prediction count does not match test-label count."
)

print("\nTest samples:", len(test_true))



# Overall test metrics


test_metrics = {
    "accuracy": accuracy_score(
        test_true,
        test_predictions
    ),

    "macro_f1": f1_score(
        test_true,
        test_predictions,
        average="macro"
    ),

    "weighted_f1": f1_score(
        test_true,
        test_predictions,
        average="weighted"
    ),
}

test_metrics_df = pd.DataFrame([test_metrics])

print("\nFINAL TEST METRICS")
display(test_metrics_df)



# Per-class classification report


report_dict = classification_report(
    test_true,
    test_predictions,
    labels=list(range(NUM_CLASSES)),
    target_names=CLASS_NAMES,
    output_dict=True,
    zero_division=0,
)

report_df = pd.DataFrame(report_dict).T

print("\nCLASSIFICATION REPORT")
display(report_df)



# Save numerical evaluation results


TEST_METRICS_PATH = (
    OUTPUT_DIR / "phase6_test_metrics.csv"
)

CLASSIFICATION_REPORT_PATH = (
    OUTPUT_DIR / "phase6_classification_report.csv"
)

test_metrics_df.to_csv(
    TEST_METRICS_PATH,
    index=False
)

report_df.to_csv(
    CLASSIFICATION_REPORT_PATH
)

print("\nSaved:", TEST_METRICS_PATH)
print("Saved:", CLASSIFICATION_REPORT_PATH)



# CONFUSION MATRIX


cm = confusion_matrix(
    test_true,
    test_predictions,
    labels=list(range(NUM_CLASSES)),
)

fig, ax = plt.subplots(
    figsize=(8, 7)
)

im = ax.imshow(cm)

ax.set_xticks(
    range(NUM_CLASSES)
)

ax.set_yticks(
    range(NUM_CLASSES)
)

ax.set_xticklabels(
    CLASS_NAMES,
    rotation=35,
    ha="right"
)

ax.set_yticklabels(
    CLASS_NAMES
)

ax.set_xlabel(
    "Predicted class"
)

ax.set_ylabel(
    "True class"
)

ax.set_title(
    "MaizeGuard — Test Confusion Matrix"
)

# Add counts inside matrix cells
for i in range(NUM_CLASSES):
    for j in range(NUM_CLASSES):

        ax.text(
            j,
            i,
            cm[i, j],
            ha="center",
            va="center"
        )

fig.colorbar(
    im,
    ax=ax
)

plt.tight_layout()

CONFUSION_PATH = (
    OUTPUT_DIR / "phase6_confusion_matrix.png"
)

plt.savefig(
    CONFUSION_PATH,
    dpi=200,
    bbox_inches="tight"
)

plt.show()

print("Saved:", CONFUSION_PATH)



# ERROR ANALYSIS


# Highest predicted probability for every image
confidence = np.max(
    test_probabilities,
    axis=1
)

error_df = pd.DataFrame({

    "index":
        np.arange(len(test_true)),

    "true_id":
        test_true,

    "true_class":
        [ID_TO_CLASS[i] for i in test_true],

    "predicted_id":
        test_predictions,

    "predicted_class":
        [ID_TO_CLASS[i] for i in test_predictions],

    "confidence":
        confidence,
})


# Mark predictions as correct or incorrect
error_df["correct"] = (
    error_df["true_id"]
    ==
    error_df["predicted_id"]
)


# Extract incorrect predictions
misclassified_df = (

    error_df.loc[
        ~error_df["correct"]
    ]

    .sort_values(
        "confidence",
        ascending=False
    )

    .reset_index(
        drop=True
    )
)


print("\nHIGHEST-CONFIDENCE MISCLASSIFICATIONS")

display(
    misclassified_df.head(20)
)



# Save prediction/error tables


ALL_PREDICTIONS_PATH = (
    OUTPUT_DIR
    / "phase6_all_test_predictions.csv"
)

MISCLASSIFIED_PATH = (
    OUTPUT_DIR
    / "phase6_misclassified_examples.csv"
)

error_df.to_csv(
    ALL_PREDICTIONS_PATH,
    index=False
)

misclassified_df.to_csv(
    MISCLASSIFIED_PATH,
    index=False
)

print(
    "\nNumber of test images:",
    len(error_df)
)

print(
    "Correct predictions:",
    error_df["correct"].sum()
)

print(
    "Misclassified images:",
    len(misclassified_df)
)

print(
    "Error rate:",
    f"{len(misclassified_df) / len(error_df):.2%}"
)



# DISPLAY HIGHEST-CONFIDENCE ERRORS


N_SHOW = min(
    12,
    len(misclassified_df)
)

if N_SHOW == 0:

    print(
        "\nNo test errors to display."
    )

else:

    ncols = 4

    nrows = int(
        np.ceil(
            N_SHOW / ncols
        )
    )

    fig, axes = plt.subplots(
        nrows,
        ncols,
        figsize=(16, 4 * nrows)
    )

    axes = np.asarray(
        axes
    ).reshape(-1)

    # Hide unused axes
    for ax in axes:
        ax.axis("off")

    # Display errors
    for ax, (_, row) in zip(
        axes,
        misclassified_df
        .head(N_SHOW)
        .iterrows()
    ):

        idx = int(
            row["index"]
        )

        image = (
            maize_dataset["test"]
            [idx]["image"]
        )

        ax.imshow(image)

        ax.set_title(
            f"True: {row['true_class']}\n"
            f"Pred: {row['predicted_class']} "
            f"({row['confidence']:.1%})"
        )

        ax.axis("off")

    plt.tight_layout()

    HIGH_CONFIDENCE_ERRORS_PATH = (
        OUTPUT_DIR
        / "phase6_high_confidence_errors.png"
    )

    plt.savefig(
        HIGH_CONFIDENCE_ERRORS_PATH,
        dpi=180,
        bbox_inches="tight"
    )

    plt.show()

    print(
        "Saved:",
        HIGH_CONFIDENCE_ERRORS_PATH
    )



# PHASE 6 SUMMARY


print("\n" + "=" * 60)
print("PHASE 6 — FINAL RESULTS")
print("=" * 60)

print(
    f"Test Accuracy:     "
    f"{test_metrics['accuracy']:.4f} "
    f"({test_metrics['accuracy']:.2%})"
)

print(
    f"Macro F1:          "
    f"{test_metrics['macro_f1']:.4f}"
)

print(
    f"Weighted F1:       "
    f"{test_metrics['weighted_f1']:.4f}"
)

print(
    f"Correct:            "
    f"{error_df['correct'].sum()} / "
    f"{len(error_df)}"
)

print(
    f"Misclassified:      "
    f"{len(misclassified_df)}"
)

print("=" * 60)

print("\nSaved Phase 6 outputs:")

print(
    "1.",
    TEST_METRICS_PATH
)

print(
    "2.",
    CLASSIFICATION_REPORT_PATH
)

print(
    "3.",
    CONFUSION_PATH
)

print(
    "4.",
    ALL_PREDICTIONS_PATH
)

print(
    "5.",
    MISCLASSIFIED_PATH
)

if N_SHOW > 0:
    print(
        "6.",
        HIGH_CONFIDENCE_ERRORS_PATH
    )

print("\nPHASE 6 COMPLETE")

Phase 7 is Explainability and the Prediction Pipeline. It adds reusable single-image prediction, class probabilities, Grad-CAM visualization, and inference metadata for later deployment.

In [ ]:

# PHASE 7 — EXPLAINABILITY AND PREDICTION PIPELINE


from pathlib import Path
from PIL import Image
import json
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf



# Load the final model selected in Phase 5


BEST_DL_MODEL_PATH = (
    MODEL_DIR / "maizeguard_best_efficientnetb0.keras"
)

if not BEST_DL_MODEL_PATH.exists():
    raise FileNotFoundError(
        f"Model not found at {BEST_DL_MODEL_PATH}. "
        "Run Phase 5 first."
    )

model = tf.keras.models.load_model(
    BEST_DL_MODEL_PATH
)

print("Loaded model:")
print(BEST_DL_MODEL_PATH)



# REUSABLE SINGLE-IMAGE PREDICTION PIPELINE


def prepare_single_image(
    image,
    image_size=IMG_SIZE
):
    """
    Prepare one maize-leaf image for EfficientNetB0 inference.

    Accepts:
        - PIL Image
        - NumPy array
        - filesystem path

    Returns:
        Tensor with shape:
        (1, IMG_HEIGHT, IMG_WIDTH, 3)

    NOTE:
        Pixel values remain in the [0,255] range because
        Keras EfficientNet preprocessing is already included
        inside the model.
    """

    if isinstance(image, (str, Path)):

        image = Image.open(
            image
        ).convert("RGB")

    elif isinstance(image, np.ndarray):

        image = Image.fromarray(
            image.astype(np.uint8)
        ).convert("RGB")

    else:

        image = image.convert("RGB")


    array = np.asarray(
        image,
        dtype=np.float32
    )

    tensor = tf.convert_to_tensor(
        array
    )

    tensor = tf.image.resize(
        tensor,
        image_size,
        antialias=True
    )

    tensor = tf.expand_dims(
        tensor,
        axis=0
    )

    return tensor


def predict_maize_leaf(image):
    """
    Predict the disease class of one maize-leaf image.

    Returns:
        predicted class
        predicted class ID
        confidence
        probability for every class
    """

    batch = prepare_single_image(
        image
    )

    probabilities = model.predict(
        batch,
        verbose=0
    )[0]

    predicted_id = int(
        np.argmax(probabilities)
    )

    result = {

        "predicted_id":
            predicted_id,

        "predicted_class":
            ID_TO_CLASS[predicted_id],

        "confidence":
            float(
                probabilities[predicted_id]
            ),

        "probabilities": {

            ID_TO_CLASS[i]:
                float(probabilities[i])

            for i in range(NUM_CLASSES)
        }
    }

    return result



# Test prediction pipeline


example = (
    maize_dataset["test"][0]["image"]
)

prediction = predict_maize_leaf(
    example
)

print("\n" + "=" * 60)
print("SINGLE-IMAGE PREDICTION")
print("=" * 60)

print(
    "\nPredicted class:",
    prediction["predicted_class"]
)

print(
    "Confidence:",
    f"{prediction['confidence']:.2%}"
)

print("\nClass probabilities:")

for class_name, probability in (
    prediction["probabilities"].items()
):

    print(
        f"{class_name:25s}: "
        f"{probability:.2%}"
    )


# Display prediction example

plt.figure(
    figsize=(6, 6)
)

plt.imshow(
    example
)

plt.title(
    f"Prediction: "
    f"{prediction['predicted_class']}\n"
    f"Confidence: "
    f"{prediction['confidence']:.1%}"
)

plt.axis("off")
plt.show()



# GRAD-CAM EXPLAINABILITY


def make_gradcam_heatmap(
    image,
    model
):
    """
    Generate a Grad-CAM heatmap using the final
    EfficientNetB0 feature maps.

    Grad-CAM indicates which image regions contributed most
    strongly to the model's prediction.
    """

    batch = prepare_single_image(
        image
    )

    # Retrieve trained model components
    augmentation_layer = model.get_layer(
        "data_augmentation"
    )

    backbone = model.get_layer(
        "efficientnetb0"
    )

    pooling_layer = model.get_layer(
        "global_average_pooling"
    )

    dropout_layer = model.get_layer(
        "dropout"
    )

    prediction_layer = model.get_layer(
        "predictions"
    )



    # Calculate gradients


    with tf.GradientTape() as tape:

        x = augmentation_layer(
            batch,
            training=False
        )

        feature_maps = backbone(
            x,
            training=False
        )

        tape.watch(
            feature_maps
        )

        x = pooling_layer(
            feature_maps
        )

        x = dropout_layer(
            x,
            training=False
        )

        predictions = prediction_layer(
            x
        )

        predicted_class = tf.argmax(
            predictions[0]
        )

        class_score = predictions[
            :,
            predicted_class
        ]


    gradients = tape.gradient(
        class_score,
        feature_maps
    )


    # Average gradient contribution
    # for each feature-map channel

    pooled_gradients = tf.reduce_mean(
        gradients,
        axis=(0, 1, 2)
    )


    feature_maps = feature_maps[0]


    # Weight feature maps according
    # to their contribution

    heatmap = tf.reduce_sum(
        feature_maps
        *
        pooled_gradients,
        axis=-1
    )


    # Keep positive contributions only

    heatmap = tf.maximum(
        heatmap,
        0
    )


    # Normalize between 0 and 1

    heatmap /= (
        tf.reduce_max(
            heatmap
        )
        +
        1e-8
    )


    return (
        heatmap.numpy(),
        int(
            predicted_class.numpy()
        )
    )



# GRAD-CAM OVERLAY


def overlay_gradcam(
    image,
    heatmap,
    alpha=0.35
):
    """
    Overlay a Grad-CAM heatmap on the original image.
    """

    if not isinstance(
        image,
        Image.Image
    ):

        if isinstance(
            image,
            (str, Path)
        ):

            image = Image.open(
                image
            ).convert("RGB")

        else:

            image = Image.fromarray(
                np.asarray(
                    image
                ).astype(
                    np.uint8
                )
            ).convert("RGB")


    original = np.asarray(
        image.convert("RGB"),
        dtype=np.float32
    )


    # Resize heatmap to original image dimensions

    heatmap_resized = tf.image.resize(

        heatmap[
            ...,
            np.newaxis
        ],

        (
            original.shape[0],
            original.shape[1]
        )

    ).numpy().squeeze()


    # Convert heatmap into color representation

    cmap = plt.get_cmap(
        "jet"
    )

    colored_heatmap = (
        cmap(
            heatmap_resized
        )[..., :3]
        *
        255.0
    )


    # Overlay heatmap on original image

    overlay = (
        (1 - alpha)
        *
        original
        +
        alpha
        *
        colored_heatmap
    )


    overlay = np.clip(
        overlay,
        0,
        255
    ).astype(
        np.uint8
    )


    return Image.fromarray(
        overlay
    )



# GENERATE GRAD-CAM EXAMPLE


heatmap, predicted_id = (
    make_gradcam_heatmap(
        example,
        model
    )
)

overlay = overlay_gradcam(
    example,
    heatmap
)



# Display original image, heatmap and overlay


fig, axes = plt.subplots(
    1,
    3,
    figsize=(15, 5)
)


axes[0].imshow(
    example
)

axes[0].set_title(
    "Original Image"
)


axes[1].imshow(
    heatmap
)

axes[1].set_title(
    "Grad-CAM Heatmap"
)


axes[2].imshow(
    overlay
)

axes[2].set_title(
    f"Prediction:\n"
    f"{ID_TO_CLASS[predicted_id]}"
)


for ax in axes:
    ax.axis("off")


plt.tight_layout()


# Save Grad-CAM visualization

GRADCAM_PATH = (
    OUTPUT_DIR
    /
    "phase7_gradcam_example.png"
)

plt.savefig(
    GRADCAM_PATH,
    dpi=200,
    bbox_inches="tight"
)

plt.show()

print(
    "\nGrad-CAM visualization saved:"
)

print(
    GRADCAM_PATH
)



# EXPORT INFERENCE METADATA


INFERENCE_CONFIG = {

    "model_name":
        "MaizeGuard EfficientNetB0",

    "image_size":
        list(IMG_SIZE),

    "class_names":
        CLASS_NAMES,

    "id_to_class":
        ID_TO_CLASS,

    "input_pixel_range":
        [0, 255],

    "notes": (
        "EfficientNet preprocessing is included in the model. "
        "Do not divide input pixels by 255 before inference."
    ),
}


INFERENCE_CONFIG_PATH = (

    METADATA_DIR
    /
    "maizeguard_inference_config.json"
)


with open(
    INFERENCE_CONFIG_PATH,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        INFERENCE_CONFIG,
        f,
        indent=2
    )



# PHASE 7 SUMMARY


print("\n" + "=" * 60)
print("PHASE 7 — EXPLAINABILITY & INFERENCE PIPELINE")
print("=" * 60)

print(
    "Prediction function: READY"
)

print(
    "Class probabilities: READY"
)

print(
    "Grad-CAM explainability: READY"
)

print(
    "Inference metadata: SAVED"
)

print(
    "\nMetadata file:"
)

print(
    INFERENCE_CONFIG_PATH
)

print(
    "\nGrad-CAM file:"
)

print(
    GRADCAM_PATH
)

print("=" * 60)

print(
    "\nPHASE 7 COMPLETE"
)

Phase 8 takes the trained MaizeGuard model and makes it usable outside the training notebook. It does three main things: converts the EfficientNetB0 model to TensorFlow Lite for lighter/portable inference, verifies that the converted model produces the expected prediction, and builds a Gradio web interface where a user can upload a maize-leaf image and receive disease probabilities. The Colab Gradio link is temporary; the generated app.py can later be used for permanent deployment.

In [ ]:

# PHASE 8 — APPLICATION DEVELOPMENT AND DEPLOYMENT


from pathlib import Path
import numpy as np
import tensorflow as tf
import gradio as gr



# 1. CHECK REQUIRED FILES


BEST_DL_MODEL_PATH = (
    MODEL_DIR / "maizeguard_best_efficientnetb0.keras"
)

if not BEST_DL_MODEL_PATH.exists():
    raise FileNotFoundError(
        f"Model not found at {BEST_DL_MODEL_PATH}. "
        "Run Phase 5 first."
    )

if not INFERENCE_CONFIG_PATH.exists():
    raise FileNotFoundError(
        f"Inference configuration not found at "
        f"{INFERENCE_CONFIG_PATH}. "
        "Run Phase 7 first."
    )

print("Required Phase 8 files found.")



# 2. LOAD TRAINED MODEL


trained_model = tf.keras.models.load_model(
    BEST_DL_MODEL_PATH
)

print(
    "Loaded trained model:",
    BEST_DL_MODEL_PATH
)



# 3. CREATE INFERENCE-ONLY MODEL


# Data augmentation is required during training,
# but it should not be part of the exported inference model.

backbone = trained_model.get_layer(
    "efficientnetb0"
)

pooling_layer = trained_model.get_layer(
    "global_average_pooling"
)

dropout_layer = trained_model.get_layer(
    "dropout"
)

prediction_layer = trained_model.get_layer(
    "predictions"
)


# Create clean inference input

export_inputs = tf.keras.Input(
    shape=(*IMG_SIZE, 3),
    name="image"
)


# Forward pass without training augmentation

x = backbone(
    export_inputs,
    training=False
)

x = pooling_layer(
    x
)

x = dropout_layer(
    x,
    training=False
)

export_outputs = prediction_layer(
    x
)


# Build inference model

inference_model = tf.keras.Model(
    inputs=export_inputs,
    outputs=export_outputs,
    name="maizeguard_inference"
)


print("\nInference-only model created.")

inference_model.summary()



# 4. CONVERT MODEL TO TENSORFLOW LITE


print(
    "\nConverting model to TensorFlow Lite..."
)

converter = tf.lite.TFLiteConverter.from_keras_model(
    inference_model
)


# Apply default optimization to reduce model size

converter.optimizations = [
    tf.lite.Optimize.DEFAULT
]


# Convert

tflite_model = converter.convert()


# Save converted model

TFLITE_MODEL_PATH = (
    MODEL_DIR
    /
    "maizeguard_efficientnetb0.tflite"
)


with open(
    TFLITE_MODEL_PATH,
    "wb"
) as f:

    f.write(
        tflite_model
    )


print("\nTensorFlow Lite model saved:")
print(TFLITE_MODEL_PATH)

print(
    "Model size:",
    f"{TFLITE_MODEL_PATH.stat().st_size / 1e6:.2f} MB"
)



# 5. VERIFY TENSORFLOW LITE INFERENCE


print(
    "\nTesting TensorFlow Lite model..."
)


# Initialize interpreter

interpreter = tf.lite.Interpreter(
    model_path=str(
        TFLITE_MODEL_PATH
    )
)

interpreter.allocate_tensors()


# Get input/output details

input_details = (
    interpreter
    .get_input_details()[0]
)

output_details = (
    interpreter
    .get_output_details()[0]
)


print(
    "TFLite input shape:",
    input_details["shape"]
)

print(
    "TFLite input dtype:",
    input_details["dtype"]
)



# Use the same example image from Phase 7


example = (
    maize_dataset["test"][0]["image"]
)


example_batch = (
    prepare_single_image(
        example
    )
    .numpy()
    .astype(
        input_details["dtype"]
    )
)


# Feed image into TFLite model

interpreter.set_tensor(
    input_details["index"],
    example_batch
)


# Run inference

interpreter.invoke()


# Retrieve probabilities

tflite_probabilities = (
    interpreter.get_tensor(
        output_details["index"]
    )[0]
)



# 6. COMPARE KERAS AND TFLITE PREDICTIONS


keras_result = predict_maize_leaf(
    example
)

keras_prediction = (
    keras_result[
        "predicted_class"
    ]
)

tflite_prediction_id = int(
    np.argmax(
        tflite_probabilities
    )
)

tflite_prediction = (
    ID_TO_CLASS[
        tflite_prediction_id
    ]
)


print("\n" + "=" * 60)
print("TFLITE VERIFICATION")
print("=" * 60)

print(
    "Keras prediction:",
    keras_prediction
)

print(
    "TFLite prediction:",
    tflite_prediction
)

print(
    "\nTFLite probabilities:"
)

for i, probability in enumerate(
    tflite_probabilities
):

    print(
        f"{ID_TO_CLASS[i]:25s}: "
        f"{probability:.2%}"
    )


if keras_prediction == tflite_prediction:

    print(
        "\n✓ Keras and TFLite predictions MATCH."
    )

else:

    print(
        "\nWARNING: Keras and TFLite predictions differ."
    )



# 7. GENERATE STANDALONE GRADIO APP


APP_PATH = (
    PROJECT_DIR
    /
    "app.py"
)


app_code = r'''
from pathlib import Path
import json
import numpy as np
from PIL import Image
import tensorflow as tf
import gradio as gr



# PATHS


BASE_DIR = Path(
    __file__
).resolve().parent


MODEL_PATH = (
    BASE_DIR
    /
    "models"
    /
    "maizeguard_best_efficientnetb0.keras"
)


CONFIG_PATH = (
    BASE_DIR
    /
    "data"
    /
    "metadata"
    /
    "maizeguard_inference_config.json"
)



# LOAD CONFIGURATION


with open(
    CONFIG_PATH,
    "r",
    encoding="utf-8"
) as f:

    config = json.load(f)


CLASS_NAMES = (
    config["class_names"]
)

IMG_SIZE = tuple(
    config["image_size"]
)



# LOAD MODEL


model = tf.keras.models.load_model(
    MODEL_PATH
)



# PREDICTION FUNCTION


def predict(image):

    if image is None:
        return {}


    image = image.convert(
        "RGB"
    )


    array = np.asarray(
        image,
        dtype=np.float32
    )


    tensor = tf.convert_to_tensor(
        array
    )


    tensor = tf.image.resize(
        tensor,
        IMG_SIZE,
        antialias=True
    )


    batch = tf.expand_dims(
        tensor,
        axis=0
    )


    probabilities = model.predict(
        batch,
        verbose=0
    )[0]


    return {

        CLASS_NAMES[i]:
            float(probabilities[i])

        for i in range(
            len(CLASS_NAMES)
        )
    }



# GRADIO INTERFACE


demo = gr.Interface(

    fn=predict,

    inputs=gr.Image(
        type="pil",
        label="Upload a maize leaf image"
    ),

    outputs=gr.Label(
        num_top_classes=4,
        label="Prediction"
    ),

    title="MaizeGuard",

    description=(
        "Upload a maize leaf photograph and MaizeGuard "
        "will classify it as Gray leaf spot, Common rust, "
        "Northern leaf blight, or Healthy. "
        "This is a prototype decision-support tool. "
        "Independent field validation is required before "
        "real-world agronomic use."
    ),
)


if __name__ == "__main__":

    demo.launch()
'''


APP_PATH.write_text(
    app_code,
    encoding="utf-8"
)


print(
    "\nStandalone Gradio application generated:"
)

print(
    APP_PATH
)



# 8. RUN GRADIO APP IN COLAB


def gradio_predict(
    image
):

    if image is None:
        return {}

    result = predict_maize_leaf(
        image
    )

    return result[
        "probabilities"
    ]


demo = gr.Interface(

    fn=gradio_predict,

    inputs=gr.Image(
        type="pil",
        label="Upload maize leaf image"
    ),

    outputs=gr.Label(
        num_top_classes=4,
        label="Prediction"
    ),

    title="MaizeGuard",

    description=(
        "Upload a maize leaf image to classify it as "
        "Gray leaf spot, Common rust, Northern leaf blight, "
        "or Healthy. "
        "This is a prototype classifier; independent field "
        "validation is required before real-world agronomic use."
    ),
)



# PHASE 8 SUMMARY


print("\n" + "=" * 60)
print("PHASE 8 — APPLICATION & DEPLOYMENT")
print("=" * 60)

print(
    "Original Keras model:",
    BEST_DL_MODEL_PATH
)

print(
    "\nTensorFlow Lite model:",
    TFLITE_MODEL_PATH
)

print(
    "\nStandalone Gradio app:",
    APP_PATH
)

print(
    "\nKeras prediction:",
    keras_prediction
)

print(
    "TFLite prediction:",
    tflite_prediction
)

print("=" * 60)

print(
    "\nLaunching MaizeGuard interface..."
)

print(
    "Upload a maize leaf image in the interface below."
)


# Launch temporary Colab application

demo.launch(
    share=True
)


print(
    "\nPHASE 8 COMPLETE"
)

Phase 9 — Real-World Validation and Documentation

This phase checks whether the model generalizes beyond the controlled PlantVillage dataset. It creates folders for real field images, evaluates those images if available, compares field performance with the Phase 6 test performance, calculates the domain gap, and creates a final MODEL_CARD.md documenting the model, results, limitations, and intended use.

In [ ]:

# PHASE 9 — REAL-WORLD VALIDATION AND DOCUMENTATION


from pathlib import Path
import pandas as pd
import numpy as np

from tqdm.auto import tqdm

from sklearn.metrics import (
    accuracy_score,
    f1_score,
    classification_report,
)



# 1. CREATE FIELD-VALIDATION FOLDER STRUCTURE


FIELD_VALIDATION_DIR = (
    DATA_DIR
    /
    "field_validation"
)

FIELD_VALIDATION_DIR.mkdir(
    parents=True,
    exist_ok=True
)


# Create one folder for each maize class

for class_name in CLASS_NAMES:

    class_dir = (
        FIELD_VALIDATION_DIR
        /
        class_name
    )

    class_dir.mkdir(
        parents=True,
        exist_ok=True
    )


print("=" * 60)
print("FIELD-VALIDATION DIRECTORY")
print("=" * 60)

print(
    "\nField-validation structure created at:"
)

print(
    FIELD_VALIDATION_DIR
)

print(
    "\nClass folders:"
)

for class_name in CLASS_NAMES:

    print(
        " -",
        FIELD_VALIDATION_DIR
        /
        class_name
    )



# 2. LOAD INDEPENDENT FIELD IMAGES


SUPPORTED_EXTENSIONS = {
    ".jpg",
    ".jpeg",
    ".png",
    ".webp"
}


field_records = []


for true_id, class_name in enumerate(
    CLASS_NAMES
):

    class_dir = (
        FIELD_VALIDATION_DIR
        /
        class_name
    )


    for image_path in sorted(
        class_dir.rglob("*")
    ):

        if (
            image_path.suffix.lower()
            not in SUPPORTED_EXTENSIONS
        ):

            continue


        field_records.append({

            "path":
                str(image_path),

            "true_id":
                true_id,

            "true_class":
                class_name,
        })


field_df = pd.DataFrame(
    field_records,
    columns=[
        "path",
        "true_id",
        "true_class"
    ]
)


print(
    f"\nFound "
    f"{len(field_df)} "
    f"field-validation images."
)


if len(field_df) > 0:

    print(
        "\nField images per class:"
    )

    display(
        field_df[
            "true_class"
        ].value_counts()
    )

else:

    print(
        "\nNo independent field images have been added yet."
    )

    print(
        "Add field photographs to the appropriate "
        "class folders and rerun Phase 9."
    )



# 3. LOAD PHASE 6 TEST METRICS IF NECESSARY


# Phase 9 compares field performance against
# the untouched PlantVillage test results from Phase 6.

if "test_metrics" not in globals():

    PHASE6_METRICS_PATH = (
        OUTPUT_DIR
        /
        "phase6_test_metrics.csv"
    )


    if PHASE6_METRICS_PATH.exists():

        phase6_metrics_df = pd.read_csv(
            PHASE6_METRICS_PATH
        )

        phase6_row = (
            phase6_metrics_df.iloc[0]
        )


        test_metrics = {

            "accuracy":
                float(
                    phase6_row[
                        "accuracy"
                    ]
                ),

            "macro_f1":
                float(
                    phase6_row[
                        "macro_f1"
                    ]
                ),

            "weighted_f1":
                float(
                    phase6_row[
                        "weighted_f1"
                    ]
                ),
        }


        print(
            "\nLoaded Phase 6 metrics:"
        )

        print(
            PHASE6_METRICS_PATH
        )


    else:

        raise FileNotFoundError(
            "Phase 6 test metrics were not found. "
            "Run Phase 6 before Phase 9."
        )



# 4. EVALUATE INDEPENDENT FIELD IMAGES


field_results_df = pd.DataFrame()

field_accuracy = None
field_macro_f1 = None


if len(field_df) == 0:

    print(
        "\nField evaluation skipped because "
        "no field images are available."
    )


else:

    print(
        "\nEvaluating independent field images..."
    )


    field_predictions = []


    for _, row in tqdm(

        field_df.iterrows(),

        total=len(field_df),

        desc="Field validation"
    ):


        result = predict_maize_leaf(
            row["path"]
        )


        field_predictions.append({

            **row.to_dict(),

            "predicted_id":
                result[
                    "predicted_id"
                ],

            "predicted_class":
                result[
                    "predicted_class"
                ],

            "confidence":
                result[
                    "confidence"
                ],
        })


    field_results_df = pd.DataFrame(
        field_predictions
    )



    # Field accuracy


    field_accuracy = accuracy_score(

        field_results_df[
            "true_id"
        ],

        field_results_df[
            "predicted_id"
        ]
    )



    # Field Macro F1


    field_macro_f1 = f1_score(

        field_results_df[
            "true_id"
        ],

        field_results_df[
            "predicted_id"
        ],

        average="macro",

        labels=list(
            range(NUM_CLASSES)
        ),

        zero_division=0
    )


    field_metrics_df = pd.DataFrame([{

        "field_accuracy":
            field_accuracy,

        "field_macro_f1":
            field_macro_f1,

        "n_field_images":
            len(
                field_results_df
            ),
    }])


    print(
        "\nFIELD-VALIDATION METRICS"
    )

    display(
        field_metrics_df
    )


    print(
        "\nFIELD CLASSIFICATION REPORT"
    )


    print(

        classification_report(

            field_results_df[
                "true_id"
            ],

            field_results_df[
                "predicted_id"
            ],

            labels=list(
                range(NUM_CLASSES)
            ),

            target_names=
                CLASS_NAMES,

            digits=4,

            zero_division=0
        )
    )



    # Identify field errors


    field_results_df[
        "correct"
    ] = (

        field_results_df[
            "true_id"
        ]
        ==
        field_results_df[
            "predicted_id"
        ]
    )


    field_errors_df = (

        field_results_df.loc[
            ~field_results_df[
                "correct"
            ]
        ]

        .sort_values(
            "confidence",
            ascending=False
        )

        .reset_index(
            drop=True
        )
    )


    print(
        "\nNumber of field errors:",
        len(field_errors_df)
    )


    if len(field_errors_df) > 0:

        print(
            "\nHighest-confidence "
            "field errors:"
        )

        display(
            field_errors_df.head(20)
        )



    # Save field results


    FIELD_PREDICTIONS_PATH = (

        OUTPUT_DIR
        /
        "phase9_field_predictions.csv"
    )


    FIELD_METRICS_PATH = (

        OUTPUT_DIR
        /
        "phase9_field_metrics.csv"
    )


    FIELD_ERRORS_PATH = (

        OUTPUT_DIR
        /
        "phase9_field_errors.csv"
    )


    field_results_df.to_csv(

        FIELD_PREDICTIONS_PATH,

        index=False
    )


    field_metrics_df.to_csv(

        FIELD_METRICS_PATH,

        index=False
    )


    field_errors_df.to_csv(

        FIELD_ERRORS_PATH,

        index=False
    )


    print(
        "\nSaved:",
        FIELD_PREDICTIONS_PATH
    )

    print(
        "Saved:",
        FIELD_METRICS_PATH
    )

    print(
        "Saved:",
        FIELD_ERRORS_PATH
    )



# 5. QUANTIFY PLANTVILLAGE-TO-FIELD DOMAIN GAP


if len(field_df) == 0:

    print(
        "\nField data required before "
        "domain-gap analysis."
    )


else:

    plantvillage_accuracy = float(
        test_metrics[
            "accuracy"
        ]
    )


    plantvillage_macro_f1 = float(
        test_metrics[
            "macro_f1"
        ]
    )


    gap_df = pd.DataFrame([

        {

            "metric":
                "accuracy",

            "plantvillage_test":
                plantvillage_accuracy,

            "field_validation":
                field_accuracy,

            "absolute_gap":
                (
                    plantvillage_accuracy
                    -
                    field_accuracy
                ),
        },

        {

            "metric":
                "macro_f1",

            "plantvillage_test":
                plantvillage_macro_f1,

            "field_validation":
                field_macro_f1,

            "absolute_gap":
                (
                    plantvillage_macro_f1
                    -
                    field_macro_f1
                ),
        }
    ])


    print(
        "\nPLANTVILLAGE → FIELD DOMAIN GAP"
    )


    display(
        gap_df
    )


    DOMAIN_GAP_PATH = (

        OUTPUT_DIR
        /
        "phase9_domain_gap.csv"
    )


    gap_df.to_csv(

        DOMAIN_GAP_PATH,

        index=False
    )


    print(
        "Saved:",
        DOMAIN_GAP_PATH
    )



# 6. GENERATE FINAL MODEL CARD


MODEL_CARD_PATH = (
    PROJECT_DIR
    /
    "MODEL_CARD.md"
)


split_sizes = {

    split:
        len(ds)

    for split, ds
    in maize_dataset.items()
}


test_accuracy_value = float(
    test_metrics[
        "accuracy"
    ]
)


test_macro_f1_value = float(
    test_metrics[
        "macro_f1"
    ]
)



# Field-validation section


if len(field_results_df) > 0:

    field_section = f"""
## Field validation

- Number of field images: {len(field_results_df)}
- Field accuracy: {field_accuracy:.4f}
- Field macro-F1: {field_macro_f1:.4f}

The field dataset was kept independent from model training
for the first reported real-world validation result.
"""


else:

    field_section = """
## Field validation

Independent real-world field validation has not yet been
completed.

Phase 9 should not be marked complete until a genuinely
independent, appropriately labeled field dataset has been
evaluated.
"""



# Final model card


model_card = f"""# MaizeGuard Model Card

## Purpose

MaizeGuard is an image-classification prototype for four
maize leaf states:

{", ".join(CLASS_NAMES)}.

## Dataset

Source: PlantVillage maize subset.

Leakage-safe split sizes:

- Train: {split_sizes["train"]}
- Validation: {split_sizes["validation"]}
- Test: {split_sizes["test"]}

The split strategy was designed to keep physical leaf
identities separate between train, validation, and test.

## Model

- Architecture: EfficientNetB0
- Initialization: ImageNet pretrained weights
- Input size: {IMG_SIZE[0]} x {IMG_SIZE[1]}
- Output classes: {NUM_CLASSES}
- Model-selection data: validation split
- Final internal evaluation data: untouched test split

## PlantVillage test performance

- Accuracy: {test_accuracy_value:.4f}
- Macro-F1: {test_macro_f1_value:.4f}

{field_section}

## Explainability

Grad-CAM is provided to visualize image regions that
influence a prediction.

Grad-CAM is a diagnostic explanation of model attention
and should not be interpreted as evidence that the
highlighted regions are biologically causal.

## Deployment

The project includes:

- A trained EfficientNetB0 model
- A TensorFlow Lite inference model
- A reusable prediction pipeline
- A Gradio application
- Inference metadata

## Limitations

- PlantVillage images are considerably more controlled
  than many real farm photographs.
- Performance may decrease under unfamiliar lighting,
  backgrounds, cameras, disease severities, cultivars,
  geographic regions, or mixed symptoms.
- The four output classes do not represent every maize
  disease, pest, nutrient deficiency, or physical injury.
- High model confidence does not guarantee correctness.
- Field performance should be evaluated independently
  before practical agricultural use.

## Intended use

Research, portfolio demonstration, machine-learning
experimentation, and prototype decision support.

## Not intended as

A replacement for agronomists, plant pathologists,
laboratory diagnosis, or locally validated crop-management
guidance.
"""


MODEL_CARD_PATH.write_text(

    model_card,

    encoding="utf-8"
)


print(
    "\n" + "=" * 60
)

print(
    "FINAL MAIZEGUARD MODEL CARD"
)

print(
    "=" * 60
)


print(
    model_card
)


print(
    "\nSaved:",
    MODEL_CARD_PATH
)



# 7. FINAL PROJECT STATUS


print(
    "\n" + "=" * 60
)

print(
    "MAIZEGUARD — FINAL PROJECT STATUS"
)

print(
    "=" * 60
)


print(
    "Phase 4: Classical ML baselines          COMPLETE"
)

print(
    "Phase 5: Deep-learning model             COMPLETE"
)

print(
    "Phase 6: Evaluation & error analysis     COMPLETE"
)

print(
    "Phase 7: Explainability & inference      COMPLETE"
)

print(
    "Phase 8: Application & deployment        COMPLETE"
)


if len(field_results_df) > 0:

    print(
        "Phase 9: Real-world validation         COMPLETE"
    )

    print(
        "\n✓ MAIZEGUARD PROJECT COMPLETE"
    )

else:

    print(
        "Phase 9: Real-world validation         PENDING FIELD DATA"
    )

    print(
        "\nPhase 9 code is ready, but the project "
        "is not yet fully validated."
    )

    print(
        "Add independent real-world maize images "
        "to the field-validation folders and rerun Phase 9."
    )


print(
    "\nFinal model card:"
)

print(
    MODEL_CARD_PATH
)

print(
    "=" * 60
)